In [1]:
# import libraries
import math
import os
import pickle
import random
import time
import itertools
import importlib

# import numpy
import numpy as np
import matplotlib.pyplot as plt


# TF12/TF13 plotting behavior: default save-only (no desktop pop-up windows)
TF12_PLOT_CFG = globals().get('TF12_PLOT_CFG', None)
if not isinstance(TF12_PLOT_CFG, dict):
    TF12_PLOT_CFG = {
        'show_figures': False,
        'close_after_draw': True,
    }
if not bool(TF12_PLOT_CFG.get('show_figures', False)):
    try:
        plt.switch_backend('Agg')
    except Exception:
        pass
    plt.ioff()
    def _tf12_no_show(*args, **kwargs):
        if bool(TF12_PLOT_CFG.get('close_after_draw', True)):
            try:
                plt.close('all')
            except Exception:
                pass
    plt.show = _tf12_no_show
import sklearn.preprocessing
from sklearn import preprocessing

import torch
import scipy
import scipy.io

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# import Koopman Libraries
from core.koopman_core_linear_GPU import KoopDNN_linear, KoopmanNet_linear, KoopmanNetCtrl_linear
from core.util import fit_standardizer
from models.koop_model import model_matricies, lift_raw, lift_scaled
from core.adapt_net_linear import AdaptNet_linear

# ===== import Frenet-real-state dynamics file =====
import dynamics.cacalv3_payload_a1 as payload_a1
import tf12_runtime as tf12_runtime

importlib.reload(payload_a1)
importlib.reload(tf12_runtime)
from dynamics.cacalv3_payload_a1 import FK_solver, single_vehicle_data_gen_multi

# control
from control_files.nmpc_osqp_ppc import NonlinearMPCController
from dynamics.learned_models_control.linear_dynamics import linear_Dynamics
from dynamics.learned_models_control.bilinear_dynamics import bilinear_Dynamics



Using device: cuda
GPU: NVIDIA GeForce RTX 5080


In [2]:

# =============================================================================
# 全局配置（所有功能都可以开关）
# =============================================================================
RUN_CFG = {
    "seed": 2026,
    "enable_koopman_training": True,
    "enable_koopman_linear_refit": True,
    "enable_four_vehicle": True,
    "enable_curved_tracking": True,
    "enable_adaptive_weight_matrix": True,
    "enable_ppc_soft_constraints": True,
    "enable_dynamic_ppc": True,
    "enable_cooperative_transport": True,
    "enable_solver_guard": True,
}

KOOPMAN_CFG = {
    "encoder_hidden_width": 128,
    "encoder_hidden_depth": 4,
    "encoder_output_dim": 24,
    "lr": 2e-4,
    "epochs": 180,
    "batch_size": 4096,
    "weight_decay": 8e-5,
    "ridge_lambda": 5e-5,
    "linear_fit_blend": 0.60,
    "previous_model_blend": 0.20,
    "deterministic_training": True,
}

MPC_CFG = {
    "path_length": 60.0,
    "horizon": 20,
    "max_sqp_iters": 2,
    "ff_gain": 0.95,
    "delta_alpha": 0.84,
    "ax_alpha": 0.82,
    "weight_smoothing": 0.82,
    "weight_rate_limit": 0.20,
    "osqp_time_limit": 0.02,
    "steer_limit": 0.24,
    "accel_limit": 4.00,
    "speed_profile_curv_gain": 12.0,
    "speed_profile_min": 1.7,
    "speed_profile_smooth": 0.85,
    "terminal_slowdown_start_s": 46.0,
    "terminal_slowdown_floor": 0.82,
    "coop_delta_clip": 0.040,
    "coop_ax_clip": 0.45,
    "coop_gain_decay_start_s": 42.0,
    "coop_gain_decay_end_s": 56.0,
    "emergency_ey_th": 0.55,
    "emergency_epsi_th": 0.22,
    "emergency_vx_cap": 2.4,
    "emergency_brake_ax": -0.85,
    "progress_guard_margin": 1e-4,
    "state_clip_ey": 1.8,
    "state_clip_epsi": 0.55,
    "state_clip_vy": 2.2,
    "state_clip_r": 1.5,
    "coop_disable_ey": 0.90,
    "coop_disable_epsi": 0.35,
    "coop_tail_scale": 0.08,
}
PPC_CFG = {
    "rho_ey_0": 0.45,
    "rho_ey_inf": 0.035,
    "lambda_ey": 1.25,
    "rho_epsi_0": 0.22,
    "rho_epsi_inf": 0.025,
    "lambda_epsi": 1.35,
    "curvature_gain": 0.45,
    "curvature_norm": 0.080,
}

FORMATION_CFG = {
    "leader_index": 0,
    "k_s": 0.08,
    "k_vx": 0.10,
    "k_ey": 0.65,
    "k_epsi": 0.25,
}


# 参考四个 notebook 融合后的 TF9 主方法配置
METHOD_CFG = {
    "name": "tf12_comm_quality_consensus_main",
    "koopman_structure": "bilinear",  # TF11默认双线性
    "use_stable_projected_A": True,
    "use_ppc": True,
    "use_dynamic_ppc": True,
    "use_adaptive_weight": True,
    "use_online_model_adaptation": True,  # TF11: 开启在线自适应动力学
    "online_adaptation_mode": "bilinear_ridge",  # TF11: 双线性+自适应
    "use_connection_compliance": True,  # TF12: 允许车辆相对货物极小位移/转角
    "use_comm_quality_consensus": True,  # TF12: 通信质量感知一致性
    "use_delay_compensation": True,      # TF12: 时延补偿预测
    "use_comm_constraint_tightening": True,
    "use_comm_degraded_fallback": True,
    "use_team_stability_guard": True,
    "use_progress_supervisor": True,
    "use_rigid_coord_correction": True,
    "comm_packet_loss_base": 0.00,
    "comm_packet_loss_gain": 0.20,
    "comm_delay_steps_max": 1,
    "comm_consensus_blend_min": 0.10,
    "comm_consensus_blend_max": 0.70,
    "comm_tighten_max_frac": 0.26,
    "comm_degrade_threshold": 0.45,
    "conn_max_rel_s": 0.070,
    "conn_max_rel_ey": 0.070,
    "conn_max_rel_psi": 0.035,
    "conn_k_s": 5.6,
    "conn_c_s": 3.8,
    "conn_k_ey": 6.6,
    "conn_c_ey": 4.2,
    "conn_k_psi": 5.0,
    "conn_c_psi": 3.4,
    "enforce_full_path": True,
    "completion_tol_s": 0.8,
    "max_extra_steps": 420,
    "horizon": 22,
    "max_sqp_iters": 2,
    "log_interval": 100,
    "realtime_mode": True,
    "realtime_budget_ratio": 0.50,
    "realtime_control_budget_sec": 0.014,
    "mpc_decimation_steps": 1,
    "min_solve_vehicles_per_step": 4,
    "mpc_skip_on_budget": False,
    "mpc_min_budget_left_sec": 8.0e-4,
    "fast_max_sqp_iters": 2,
    "fast_time_limit": 6.0e-3,
    "fast_time_limit_min": 1.5e-3,
    "vehicle_order_leader_first": True,
    "realtime_adapt_stride": 4,
    "realtime_disable_online_adapt": False,
    "realtime_horizon": 18,
    "realtime_disable_gc": True,
    "leader_always_solve": True,
    "time_limit": 0.02,
    "progress_lag_activate_s": 0.03,
    "progress_lag_full_s": 0.30,
    "progress_recover_ax_min": 0.80,
    "progress_recover_ax_max": 3.50,
    "emergency_override_lag_s": 0.8,
    "emergency_override_ax_min": 0.3,
    "emergency_override_ax_max": 2.5,

}
ADAPT_CFG = {
    # 参考 Serial / Quadrotor: 在线 AdaptNet_linear + 低跳变约束
    "window": 18,
    "update_every": 6,
    "forget_factor": 0.97,
    "ridge_lambda": 2e-4,
    "max_delta_a_norm": 0.06,
    "max_delta_b_norm": 0.06,
    "model_update_blend": 0.04,
    "project_after_update": True,
    "project_radius": 0.998,

    # 仅用高置信样本更新，避免发散车辆污染模型
    "leader_only_samples": True,
    "max_sample_ey": 0.70,
    "max_sample_epsi": 0.35,
    "max_sample_vy": 1.20,
    "max_sample_r": 1.20,
    "max_sample_dz_norm": 1.60,
    "adapt_stop_s": 22.0,

    # AdaptNet_linear 参数（来自参考 notebook）
    "net_lr": 1e-4,
    "net_epochs": 4,
    "net_batch_size": 16,
    "net_l1_reg": 3e-3,
    "net_l2_reg": 3e-3,
    "net_optimizer": "adam",
    "net_warm_start": True,
}
from control_files.tf12.seed_utils import set_global_seed

set_global_seed(RUN_CFG["seed"], deterministic=KOOPMAN_CFG["deterministic_training"])
print("功能开关:", RUN_CFG)
print("TF12主方法配置:", METHOD_CFG)


# =============================================================================
# TF12：主方法模块配置（A1 notebook stable path）
# =============================================================================
TF11_MODULES_MAIN = {
    # 结构相关（TF11主方法核心）
    "stable_projected_A": True,
    "koopman_structure": "bilinear",  # linear | bilinear

    # 控制策略相关
    "adaptive_weight": True,
    "ppc": True,
    "dynamic_ppc": True,
    "cooperative_transport": True,
    "online_adaptation": True,   # TF11: 保留在线自适应动力学

    # 稳定性保护相关
    "emergency_guard": True,
    "progress_guard": True,
    "solver_guard": True,
}

TF12_MODULES_MAIN = dict(TF11_MODULES_MAIN)

# 兼容后续函数命名（run_tf10_case 仍复用）
TF10_MODULES_DEFAULT = dict(TF11_MODULES_MAIN)

# TF11仅保留主方法，不再做消融循环
TF10_ABLATION_CASES = []

TF10_RUN_PLAN = {
    "run_main": True,
    "run_ablation": False,
    "ablation_max_cases": 0,
    "ablation_log": False,
}

# =============================================================================
# A1 rigid-payload configuration
# =============================================================================
A1_PAYLOAD_CFG = payload_a1.build_payload_config(payload_mass=2000.0, payload_length=5.0, payload_width=2.0, com_height=1.2)
A1_VARIATION_SUBSET_WEIGHTS = {1: 0.40, 2: 0.30, 3: 0.20, 4: 0.10}
A1_MAIN_CHANGE_MASK = (0, 1, 2, 3)
A1_DATASET_TAG = "a1_rigid_payload_2t"





功能开关: {'seed': 2026, 'enable_koopman_training': True, 'enable_koopman_linear_refit': True, 'enable_four_vehicle': True, 'enable_curved_tracking': True, 'enable_adaptive_weight_matrix': True, 'enable_ppc_soft_constraints': True, 'enable_dynamic_ppc': True, 'enable_cooperative_transport': True, 'enable_solver_guard': True}
TF12主方法配置: {'name': 'tf12_comm_quality_consensus_main', 'koopman_structure': 'bilinear', 'use_stable_projected_A': True, 'use_ppc': True, 'use_dynamic_ppc': True, 'use_adaptive_weight': True, 'use_online_model_adaptation': True, 'online_adaptation_mode': 'bilinear_ridge', 'use_connection_compliance': True, 'use_comm_quality_consensus': True, 'use_delay_compensation': True, 'use_comm_constraint_tightening': True, 'use_comm_degraded_fallback': True, 'use_team_stability_guard': True, 'use_progress_supervisor': True, 'use_rigid_coord_correction': True, 'comm_packet_loss_base': 0.0, 'comm_packet_loss_gain': 0.2, 'comm_delay_steps_max': 1, 'comm_consensus_blend_min': 0.1, 'c

In [3]:
# =========================
# TF12 stable run hygiene
# =========================
A1_NOTEBOOK_VERSION = "tf12_stable_2026_04_17"

A1_STALE_GLOBALS = [
    "A1_COORDINATOR",
    "A1_REF_BUNDLE",
    "A1_REFERENCE_VERSION",
    "A1_RUNTIME_CTX",
    "A1_DATA",
    "main_result",
    "compare_results",
    "payload_force_hist",
    "payload_force_summary",
    "payload_change_mask",
    "ref_team_hist",
    "ref_vehicle_histories",
    "ref_leader_relative_targets",
    "team_state_hist",
    "team_input_hist",
    "xt_actual_vehicles",
    "u_vehicles",
    "z_vehicles",
]
for _name in A1_STALE_GLOBALS:
    globals().pop(_name, None)

import control_files.rigid_payload_coordinator_a1 as rigid_payload_coordinator_a1
import control_files.rigid_payload_stability_guard_a1 as rigid_payload_stability_guard_a1

payload_a1 = importlib.reload(payload_a1)
rigid_payload_coordinator_a1 = importlib.reload(rigid_payload_coordinator_a1)
rigid_payload_stability_guard_a1 = importlib.reload(rigid_payload_stability_guard_a1)
tf12_runtime = importlib.reload(tf12_runtime)

print("[A1] notebook hygiene reset complete:", A1_NOTEBOOK_VERSION)
print("[A1] stale globals cleared:", len(A1_STALE_GLOBALS))
print("[A1] recommendation: restart kernel and run the notebook from top to bottom.")


[A1] notebook hygiene reset complete: tf12_stable_2026_04_17
[A1] stale globals cleared: 18
[A1] recommendation: restart kernel and run the notebook from top to bottom.


In [4]:

# ===== helper imports for path/state/Koopman =====
import importlib
import control_files.tf12.core_utils as tf12_core

tf12_core = importlib.reload(tf12_core)
tf12_core.configure_fk_solver(FK_solver)

build_test_frenet_path_from_xy = tf12_core.build_test_frenet_path_from_xy
frenet_to_global = tf12_core.frenet_to_global
to_numpy = tf12_core.to_numpy

scale_state = tf12_core.scale_state
scale_state_batch = tf12_core.scale_state_batch
decode_scaled_to_raw = tf12_core.decode_scaled_to_raw

clip_closed_loop_state = tf12_core.clip_closed_loop_state
is_finite_vector = tf12_core.is_finite_vector
safe_FK_step = tf12_core.safe_FK_step

_lift_batch_with_net = tf12_core._lift_batch_with_net
fit_koopman_linear_matrices = tf12_core.fit_koopman_linear_matrices
blend_matrix = tf12_core.blend_matrix
summarize_solver_status = tf12_core.summarize_solver_status


In [5]:
# =============================================================================
# System identification for A1 rigid payload transport
# =============================================================================
linear = True
a1_data = tf12_runtime.generate_a1_dataset(
    run_cfg=RUN_CFG,
    koopman_cfg=KOOPMAN_CFG,
    set_global_seed_fn=set_global_seed,
    payload_module=payload_a1,
    single_vehicle_data_gen_multi_fn=single_vehicle_data_gen_multi,
    payload_cfg=A1_PAYLOAD_CFG,
    subset_weights=A1_VARIATION_SUBSET_WEIGHTS,
    dataset_tag=A1_DATASET_TAG,
)
A1_DATA = dict(a1_data)

X = A1_DATA["X"]
X_changed = A1_DATA["X_changed"]
U = A1_DATA["U"]
num_states = A1_DATA["num_states"]
num_inputs = A1_DATA["num_inputs"]
dt = A1_DATA["dt"]
sys_pars_base = A1_DATA["sys_pars_base"]
sys_pars_new_base = A1_DATA["sys_pars_new_base"]
sys_pars = A1_DATA["sys_pars"]
sys_pars_new = A1_DATA["sys_pars_new"]
sensor_noise = A1_DATA["sensor_noise"]
SNR_DB = A1_DATA["SNR_DB"]
train_path_length_target = A1_DATA["train_path_length_target"]
num_snaps = A1_DATA["num_snaps"]
num_traj = A1_DATA["num_traj"]
num_train = A1_DATA["num_train"]
num_val = A1_DATA["num_val"]
variation_labels_all = A1_DATA["variation_labels_all"]
variation_summary = A1_DATA["variation_summary"]
dataset_npz = A1_DATA["dataset_npz"]
dataset_pars = A1_DATA["dataset_pars"]

print("[A1] dataset assignment mode: explicit (no globals().update)")
print("X shape:", X.shape)
print("X_changed shape:", X_changed.shape)
print("U shape:", U.shape)
print("Variation summary:", variation_summary)

state_labels = ["s", "e_y", "e_psi", "v_x", "v_y", "r"]
t_train = np.linspace(0, dt * (num_snaps - 1), num_snaps)

plt.figure(figsize=(18, 12))
for j in range(min(num_traj, 24)):
    for i in range(num_states):
        plt.subplot(num_states, 1, i + 1)
        plt.plot(t_train, X[j, :, i], alpha=0.55)
        plt.ylabel(state_labels[i])
        plt.xlabel("t")
plt.suptitle("A1 Nominal Corner Dynamics Under Rigid Payload", fontsize=18)
plt.tight_layout()
plt.show()

plt.figure(figsize=(18, 12))
for j in range(min(num_traj, 24)):
    for i in range(num_states):
        plt.subplot(num_states, 1, i + 1)
        plt.plot(t_train, X_changed[j, :, i], alpha=0.55)
        plt.ylabel(state_labels[i])
        plt.xlabel("t")
plt.suptitle("A1 Changed Corner Dynamics With Vehicle-Combination Variations", fontsize=18)
plt.tight_layout()
plt.show()


[A1] dataset assignment mode: explicit (no globals().update)
X shape: (720, 1400, 6)
X_changed shape: (720, 1400, 6)
U shape: (720, 1399, 2)
Variation summary: {'veh_1': 100, 'veh_1_2': 40, 'veh_1_2_3': 80, 'veh_1_2_3_4': 20, 'veh_1_3': 40, 'veh_1_3_4': 20, 'veh_1_4': 60, 'veh_2': 120, 'veh_2_3': 20, 'veh_2_4': 40, 'veh_3_4': 140, 'veh_4': 40}


In [6]:
# =============================================================================
# Learning Koopman
# =============================================================================
xs_train, us_train = X[:num_train, :, :], U[:num_train, :, :]
xs_val, us_val = X[num_train:, :, :], U[num_train:, :, :]

net_params_lin = {}
net_params_lin["state_dim"] = num_states
net_params_lin["ctrl_dim"] = num_inputs
net_params_lin["encoder_hidden_width"] = KOOPMAN_CFG["encoder_hidden_width"]
net_params_lin["encoder_hidden_depth"] = KOOPMAN_CFG["encoder_hidden_depth"]
net_params_lin["encoder_output_dim"] = KOOPMAN_CFG["encoder_output_dim"]
net_params_lin["optimizer"] = "adam"
net_params_lin["activation_type"] = "gelu"
net_params_lin["lr"] = KOOPMAN_CFG["lr"]
net_params_lin["epochs"] = KOOPMAN_CFG["epochs"]
net_params_lin["batch_size"] = KOOPMAN_CFG["batch_size"]

net_params_lin["loss_mode"] = "absolute"
net_params_lin["delta_loss_use_scale"] = True
net_params_lin["delta_scale_min"] = 1e-2
net_params_lin["log_process_align"] = False

net_params_lin["eig_loss"] = True
net_params_lin["eig_loss_coeff"] = 0.01
net_params_lin["lifted_loss_penalty"] = 0.60

net_params_lin["l2_reg"] = 8e-5
net_params_lin["l1_reg"] = 0.0
net_params_lin["first_obs_const"] = True
net_params_lin["override_C"] = False
net_params_lin["dt"] = dt
net_params_lin["weight_decay"] = KOOPMAN_CFG["weight_decay"]

train = RUN_CFG["enable_koopman_training"]
standardize = True

os.makedirs("saved_models/single_vehicle/linear", exist_ok=True)

file_koop_linear = (
        "saved_models/single_vehicle/linear/"
        + "Koop_vehicle_Dim"
        + str(net_params_lin["encoder_output_dim"])
        + "_dt_"
        + str(dt)
        + "_tf12b1_tuned_abs_learnC.pth"
)

print("Model save path:", file_koop_linear)

standardizer_u_kdnn = fit_standardizer(
    us_train, preprocessing.StandardScaler(with_mean=True)
)
standardizer_x_kdnn = fit_standardizer(
    xs_train, preprocessing.StandardScaler(with_mean=True)
)

set_global_seed(RUN_CFG["seed"], deterministic=KOOPMAN_CFG["deterministic_training"])
torch.cuda.empty_cache()

prev_A = prev_B = None
if RUN_CFG["enable_koopman_linear_refit"] and os.path.exists(file_koop_linear):
    try:
        prev_model = torch.load(file_koop_linear, map_location="cpu", weights_only=False)
        if not hasattr(prev_model, "A_lin") or not hasattr(prev_model, "B_lin"):
            prev_model.construct_koopman_model()
        prev_A = np.array(prev_model.A_lin, dtype=np.float32)
        prev_B = np.array(prev_model.B_lin, dtype=np.float32)
        print("检测到历史模型，将用于平滑本次线性矩阵。")
    except Exception as e:
        print("历史模型读取失败，跳过历史平滑:", e)

if train:
    if standardize:
        net = KoopmanNetCtrl_linear(
            net_params_lin,
            standardizer_x=standardizer_x_kdnn,
            standardizer_u=standardizer_u_kdnn,
            device=DEVICE
        )
    else:
        net = KoopmanNetCtrl_linear(
            net_params_lin,
            device=DEVICE
        )

    model_koop_dnn_lin = KoopDNN_linear(net)

    # 强制对齐：x 和 u 的时间维以最小长度为准
    min_t_train = min(xs_train.shape[1], us_train.shape[1])
    xs_train = xs_train[:, :min_t_train, :]
    us_train = us_train[:, :min_t_train, :]

    min_t_val = min(xs_val.shape[1], us_val.shape[1])
    xs_val = xs_val[:, :min_t_val, :]
    us_val = us_val[:, :min_t_val, :]

    model_koop_dnn_lin.set_datasets(
        xs_train,
        u_train=us_train,
        x_val=xs_val,
        u_val=us_val
    )

    X_train, y_train = model_koop_dnn_lin.net.process(
        model_koop_dnn_lin.x_train, data_u=model_koop_dnn_lin.u_train
    )
    X_val, y_val = model_koop_dnn_lin.net.process(
        model_koop_dnn_lin.x_val, data_u=model_koop_dnn_lin.u_val, train_mode=False
    )

    from torch.utils.data import TensorDataset, DataLoader

    X_train_t = torch.from_numpy(X_train).float()
    y_train_t = torch.from_numpy(y_train).float()
    X_val_t = torch.from_numpy(X_val).float()
    y_val_t = torch.from_numpy(y_val).float()

    train_dataset = TensorDataset(X_train_t, y_train_t)
    val_dataset = TensorDataset(X_val_t, y_val_t)

    loader_generator = torch.Generator()
    loader_generator.manual_seed(RUN_CFG["seed"])

    train_loader = DataLoader(
        train_dataset,
        batch_size=net_params_lin["batch_size"],
        shuffle=True,
        num_workers=0,
        pin_memory=True,
        drop_last=True,
        persistent_workers=False,
        prefetch_factor=None,
        generator=loader_generator
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=net_params_lin["batch_size"],
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    model_koop_dnn_lin.train_loader = train_loader
    model_koop_dnn_lin.val_loader = val_loader

    model_koop_dnn_lin.model_pipeline(net_params_lin, print_epoch=True)
else:
    model_koop_dnn_lin = torch.load(file_koop_linear, map_location="cpu", weights_only=False)
    model_koop_dnn_lin.net.to(DEVICE)
    model_koop_dnn_lin.net.device = DEVICE

# 始终构造一次线性 Koopman 矩阵（供 MPC 使用）
model_koop_dnn_lin.construct_koopman_model()

if RUN_CFG["enable_koopman_linear_refit"]:
    A_fit, B_fit, fit_info = fit_koopman_linear_matrices(
        model_koop_dnn_lin,
        xs_train,
        us_train,
        ridge_lambda=KOOPMAN_CFG["ridge_lambda"],
        batch_size=8192
    )

    A_refined = blend_matrix(model_koop_dnn_lin.A_lin.astype(np.float32), A_fit, KOOPMAN_CFG["linear_fit_blend"])
    B_refined = blend_matrix(model_koop_dnn_lin.B_lin.astype(np.float32), B_fit, KOOPMAN_CFG["linear_fit_blend"])

    if prev_A is not None and prev_B is not None:
        A_refined = blend_matrix(A_refined, prev_A, KOOPMAN_CFG["previous_model_blend"])
        B_refined = blend_matrix(B_refined, prev_B, KOOPMAN_CFG["previous_model_blend"])

    model_koop_dnn_lin.A_lin = A_refined
    model_koop_dnn_lin.B_lin = B_refined

    with torch.no_grad():
        model_koop_dnn_lin.net.A.weight.copy_(torch.from_numpy(A_refined).to(DEVICE))
        model_koop_dnn_lin.net.B.weight.copy_(torch.from_numpy(B_refined).to(DEVICE))

    print(f"线性重拟合完成: samples={fit_info['samples']}, lifted_rmse={fit_info['lifted_rmse']:.4e}")

if train:
    torch.save(model_koop_dnn_lin, file_koop_linear)

if len(model_koop_dnn_lin.train_loss_hist) > 0 and len(model_koop_dnn_lin.val_loss_hist) > 0:
    train_loss = [l[0] for l in model_koop_dnn_lin.train_loss_hist]
    train_pred_loss = [l[1] for l in model_koop_dnn_lin.train_loss_hist]
    train_lifted_loss = [l[2] for l in model_koop_dnn_lin.train_loss_hist]
    val_loss = [l[0] for l in model_koop_dnn_lin.val_loss_hist]
    val_pred_loss = [l[1] for l in model_koop_dnn_lin.val_loss_hist]
    val_lifted_loss = [l[2] for l in model_koop_dnn_lin.val_loss_hist]
    epochs = np.arange(0, len(train_loss))

    plt.figure(figsize=(15, 8))
    plt.plot(epochs, train_loss, color="tab:orange", label="Training loss")
    plt.plot(epochs, train_pred_loss, "--", color="tab:orange", label="Training prediction loss")
    plt.plot(epochs, train_lifted_loss, ":", color="tab:orange", label="Training lifted loss")
    plt.plot(epochs, val_loss, color="tab:blue", label="Validation loss")
    plt.plot(epochs, val_pred_loss, "--", color="tab:blue", label="Validation prediction loss")
    plt.plot(epochs, val_lifted_loss, ":", color="tab:blue", label="Validation lifted loss")
    plt.legend()
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.yscale("log")
    plt.show()
else:
    print("未执行本轮训练，跳过训练曲线绘制。")


Model save path: saved_models/single_vehicle/linear/Koop_vehicle_Dim24_dt_0.02_tf12b1_tuned_abs_learnC.pth
检测到历史模型，将用于平滑本次线性矩阵。
Epoch [  10/180] | Train  total=1.2658e-02  pred=8.1528e-04  lifted=4.0296e-03| Val    total=1.2007e-02  pred=7.9893e-04  lifted=3.6725e-03
Epoch [  20/180] | Train  total=7.7260e-03  pred=6.5280e-04  lifted=4.2944e-03| Val    total=7.6025e-03  pred=6.5596e-04  lifted=4.3340e-03
Epoch [  30/180] | Train  total=5.7621e-03  pred=5.0940e-04  lifted=5.7551e-03| Val    total=6.1613e-03  pred=5.2696e-04  lifted=6.6284e-03
Epoch [  40/180] | Train  total=1.8492e-03  pred=2.9264e-04  lifted=2.2827e-03| Val    total=1.8487e-03  pred=3.0169e-04  lifted=2.3044e-03
Epoch [  50/180] | Train  total=6.4661e-04  pred=1.8833e-04  lifted=7.4396e-04| Val    total=6.3199e-04  pred=2.0356e-04  lifted=6.9716e-04
Epoch [  60/180] | Train  total=3.3035e-04  pred=1.7657e-04  lifted=2.5517e-04| Val    total=3.3335e-04  pred=1.9304e-04  lifted=2.3281e-04
Epoch [  70/180] | Train  total=

KeyboardInterrupt: 

In [ ]:
# =============================================================================
# Koopman Model Parameters (Open Loop Test)
# =============================================================================
num_snaps_test = 100
T_test = np.linspace(0, (num_snaps_test - 1) * dt, num_snaps_test)

num_traj_test = 1
first_obs_const = int(net_params_lin["first_obs_const"])
override_C = net_params_lin["override_C"]
# 统一以当前模型矩阵维度为准，避免 override_C=False 时观测维度计算不一致
if not hasattr(model_koop_dnn_lin, 'A_lin'):
    model_koop_dnn_lin.construct_koopman_model()
n_obs_lin = int(model_koop_dnn_lin.A_lin.shape[0])

x_unchanged_test, x_changed_test, u_test = single_vehicle_data_gen_multi(
    num_traj_test, num_snaps_test, sys_pars, sys_pars_new, sensor_noise, SNR_DB
)

# A1 数据生成器会返回四个角点车辆的轨迹，因此开环评估必须明确选定一辆车。
open_loop_vehicle_idx = 0
x_unchanged_eval = x_unchanged_test[open_loop_vehicle_idx:open_loop_vehicle_idx + 1, :, :]
x_changed_eval = x_changed_test[open_loop_vehicle_idx:open_loop_vehicle_idx + 1, :, :]
u_eval = u_test[open_loop_vehicle_idx:open_loop_vehicle_idx + 1, :, :]

print("open-loop eval vehicle idx:", open_loop_vehicle_idx)
print("x_unchanged_test shape:", x_unchanged_test.shape)
print("u_test shape:", u_test.shape)
print("x_unchanged_eval shape:", x_unchanged_eval.shape)
print("u_eval shape:", u_eval.shape)

# ====================== 正确提取 Koopman 矩阵 ======================
if not hasattr(model_koop_dnn_lin, 'A_lin'):
    print("尚未调用 construct_koopman_model()，现在自动调用...")
    model_koop_dnn_lin.construct_koopman_model()

A_lin = model_koop_dnn_lin.A_lin
B_lin = model_koop_dnn_lin.B_lin
C_lin = model_koop_dnn_lin.C_np

print("A, B, C shapes:", A_lin.shape, B_lin.shape, C_lin.shape)

eigvals, eigvecs = np.linalg.eig(A_lin)
eigvals_proj = np.array([
    ev if np.abs(ev) <= 0.999 else ev / np.abs(ev) * 0.999
    for ev in eigvals
])
A_lin_stable = eigvecs @ np.diag(eigvals_proj) @ np.linalg.inv(eigvecs)
A_lin_stable = np.real(A_lin_stable)

eigvals_stable = np.linalg.eigvals(A_lin_stable)
print("stable spectral radius =", np.max(np.abs(eigvals_stable)))

spectral_radius = np.max(np.abs(eigvals))
print("spectral radius =", spectral_radius)
print("eigvals =", eigvals)

plt.figure(figsize=(6, 6))
plt.scatter(np.real(eigvals), np.imag(eigvals), label="eig(A)")
theta = np.linspace(0, 2 * np.pi, 400)
plt.plot(np.cos(theta), np.sin(theta), "--", label="unit circle")
plt.axhline(0)
plt.axvline(0)
plt.xlabel("Real")
plt.ylabel("Imag")
plt.legend()
plt.title("Eigenvalues of A_lin")
plt.axis("equal")
plt.show()

X_unchanged_scaled, _ = model_koop_dnn_lin.net.process(x_unchanged_eval, data_u=u_eval)
X_changed_scaled, _ = model_koop_dnn_lin.net.process(x_changed_eval, data_u=u_eval)

x_unchanged_scaled = X_unchanged_scaled[:, :num_states]
u_scaled = X_unchanged_scaled[:, num_states:num_states + num_inputs]
x_unchanged_prime_scaled = X_unchanged_scaled[:, num_states + num_inputs:]

x_changed_scaled = X_changed_scaled[:, :num_states]
x_changed_prime_scaled = X_changed_scaled[:, num_states + num_inputs:]

print("x_unchanged_scaled shape:", x_unchanged_scaled.shape)
print("u_scaled shape:", u_scaled.shape)
print("x_changed_scaled shape:", x_changed_scaled.shape)
print("x_changed_prime_scaled shape:", x_changed_prime_scaled.shape)

num_pred = min(x_unchanged_scaled.shape[0], u_scaled.shape[0], x_unchanged_eval.shape[1] - 1)
print("open-loop effective num_pred =", num_pred)

x_unchanged_scaled = x_unchanged_scaled[:num_pred, :]
u_scaled = u_scaled[:num_pred, :]
x_changed_scaled = x_changed_scaled[:num_pred, :]
if x_unchanged_prime_scaled.size > 0:
    x_unchanged_prime_scaled = x_unchanged_prime_scaled[:num_pred, :]
if x_changed_prime_scaled.size > 0:
    x_changed_prime_scaled = x_changed_prime_scaled[:num_pred, :]

z_lin = np.zeros((num_pred, n_obs_lin))
x_est_lin = np.zeros((num_pred, num_states))

z_lin[0, :] = lift_scaled(x_unchanged_scaled[0, :], model_koop_dnn_lin, net_params_lin)
x_est_lin[0, :] = np.matmul(z_lin[0, :], C_lin.T)

for k in range(num_pred - 1):
    z_lin[k + 1, :] = np.matmul(z_lin[k, :], A_lin.T) + np.matmul(u_scaled[k, :], B_lin.T)
    x_est_lin[k + 1, :] = np.matmul(z_lin[k + 1, :], C_lin.T)

z_lin_stable = np.zeros((num_pred, n_obs_lin))
x_est_lin_stable = np.zeros((num_pred, num_states))

z_lin_stable[0, :] = lift_scaled(x_unchanged_scaled[0, :], model_koop_dnn_lin, net_params_lin)
x_est_lin_stable[0, :] = np.matmul(z_lin_stable[0, :], C_lin.T)

for k in range(num_pred - 1):
    z_lin_stable[k + 1, :] = np.matmul(z_lin_stable[k, :], A_lin_stable.T) + np.matmul(u_scaled[k, :], B_lin.T)
    x_est_lin_stable[k + 1, :] = np.matmul(z_lin_stable[k + 1, :], C_lin.T)

T_plot = np.arange(num_pred) * dt

plt.figure(figsize=(18, 12))
for i in range(num_states):
    plt.subplot(num_states, 1, i + 1)
    plt.xlabel("t")
    plt.ylabel(state_labels[i])
    plt.plot(T_plot, x_unchanged_scaled[:, i], label="True")
    plt.plot(T_plot, x_est_lin[:, i], "--", label="Open-loop Koopman")
    plt.plot(T_plot, x_est_lin_stable[:, i], "-.", label="Stable-projected Koopman")
    plt.legend()

plt.suptitle("Open-loop Prediction Comparison (scaled state space)", fontsize=20)
plt.tight_layout()
plt.show()

x_est_lin_raw = standardizer_x_kdnn.inverse_transform(x_est_lin)
x_est_lin_stable_raw = standardizer_x_kdnn.inverse_transform(x_est_lin_stable)
x_true_raw = x_unchanged_eval[0, :num_pred, :]

assert x_true_raw.shape[0] == num_pred
assert x_est_lin_raw.shape[0] == num_pred
assert x_est_lin_stable_raw.shape[0] == num_pred

plt.figure(figsize=(18, 12))
for i in range(num_states):
    plt.subplot(num_states, 1, i + 1)
    plt.xlabel("t")
    plt.ylabel(state_labels[i])
    plt.plot(T_plot, x_true_raw[:, i], label="True (raw)")
    plt.plot(T_plot, x_est_lin_raw[:, i], "--", label="Open-loop Koopman (raw)")
    plt.plot(T_plot, x_est_lin_stable_raw[:, i], "-.", label="Stable-projected Koopman (raw)")
    plt.legend()

plt.suptitle("Open-loop Prediction Comparison (raw state space)", fontsize=20)
plt.tight_layout()
plt.show()


In [ ]:
# ====================== One-step Prediction（按单车测试轨迹对齐） ======================
num_pred_1 = min(x_unchanged_scaled.shape[0], u_scaled.shape[0], x_unchanged_eval.shape[1] - 1)
x_true_next_raw = x_unchanged_eval[0, 1:1 + num_pred_1, :]
x_true_next_scaled = standardizer_x_kdnn.transform(x_true_next_raw)

x_one_step_pred = np.zeros((num_pred_1, num_states))

print(f"进行 one-step 预测 | 样本数: {num_pred_1} | 状态维: {num_states}")

for k in range(num_pred_1):
    z_k = lift_scaled(x_unchanged_scaled[k, :], model_koop_dnn_lin, net_params_lin)
    z_k1 = np.matmul(z_k, A_lin.T) + np.matmul(u_scaled[k, :], B_lin.T)
    x_one_step_pred[k, :] = np.matmul(z_k1, C_lin.T)

T_plot_1 = np.arange(num_pred_1) * dt

# ====================== 绘图（scaled） ======================
plt.figure(figsize=(18, 12))
for i in range(num_states):
    plt.subplot(num_states, 1, i + 1)
    plt.plot(T_plot_1, x_true_next_scaled[:, i], label="True next state")
    plt.plot(T_plot_1, x_one_step_pred[:, i], "--", label="One-step Koopman")
    plt.ylabel(state_labels[i])
    plt.legend()
plt.suptitle("One-step Prediction Performance (scaled state space)", fontsize=20)
plt.tight_layout()
plt.show()

# ====================== 绘图（raw） ======================
x_one_step_pred_raw = standardizer_x_kdnn.inverse_transform(x_one_step_pred)

assert x_true_next_raw.shape[0] == num_pred_1
assert x_one_step_pred_raw.shape[0] == num_pred_1

T_plot_raw = np.arange(num_pred_1) * dt

plt.figure(figsize=(18, 12))
for i in range(num_states):
    plt.subplot(num_states, 1, i + 1)
    plt.plot(T_plot_raw, x_true_next_raw[:, i], label="True next state (raw)")
    plt.plot(T_plot_raw, x_one_step_pred_raw[:, i], "--", label="One-step Koopman (raw)")
    plt.ylabel(state_labels[i])
    plt.legend()
plt.suptitle("One-step Prediction Performance (raw state space)", fontsize=20)
plt.tight_layout()
plt.show()


In [ ]:
#
#=============================================================================
# Testing - TF12 rigid payload transport baseline
# =============================================================================

# =========================
# Four vehicles at the rectangle corners
# =========================
num_vehicles = 4 if RUN_CFG["enable_four_vehicle"] else 1
vx_nom = 3.0
x0_payload_center, formation_offsets, x0_vehicles = tf12_runtime.build_a1_initial_states(payload_a1, A1_PAYLOAD_CFG, vx_nom)

# =========================
# Reference path generation (DLC + Hairpin library)
# =========================
TF12_PATH_CFG = globals().get("TF12_PATH_CFG", None)
if not isinstance(TF12_PATH_CFG, dict):
    TF12_PATH_CFG = {
        "active_mode": "dlc",          # dlc | hairpin
        "build_hairpin": True,
        "plot_path_library": True,

        # DLC params: 降曲率/降速度，减小跟踪误差
        "dlc_amp_1": 0.85,
        "dlc_amp_2": -0.85,
        "dlc_sigma_ratio": 0.16,
        "dlc_speed_cap": 2.40,

        # Hairpin params: 开始/结束各40m直线 + 尺寸约束半径
        "hairpin_vehicle_length": 4.4,
        "hairpin_vehicle_width": 1.9,
        "hairpin_clearance": 1.20,
        "hairpin_radius_override": 50.0,   # 默认采用可跟踪性更好的中等半径
        "hairpin_radius_safety_factor": 1.20,
        "hairpin_shape_mode": "right_angle_pair",
        "force_right_angle_pair": True,
        "hairpin_entry_straight": 1.0,
        "hairpin_top_straight": 8.0,
        "hairpin_red_bulge_x": 38.0,
        "hairpin_red_height": 40.0,
        "hairpin_red_curvature_safe": False,
        # 类直角双弯（蓝线风格）参数
        "hairpin_ra_right_x": 33.0,
        "hairpin_ra_bottom_y": 2.5,
        "hairpin_ra_top_y": 16.0,
        "hairpin_ra_bottom_x0": 0.0,
        "hairpin_ra_top_x0": 0.0,
        "hairpin_ra_corner_r": 12.0,
        "hairpin_ra_bottom_rise_len": 12.0,
        "hairpin_ra_bottom_rise_h": 1.6,
        "hairpin_ra_top_drop_len": 12.0,
        "hairpin_ra_top_drop_h": 1.6,
        "hairpin_ra_curvature_safe": True,
        "hairpin_ra_side_wave_amp": 0.01,
        "hairpin_ra_side_wave_harm": 1.0,
        "hairpin_ra_top_wave_amp": 0.008,
        "hairpin_ra_bottom_wave_amp": 0.008,
        "hairpin_ra_smooth_win": 45,
        "hairpin_ra_smooth_pass": 6,
        "hairpin_kappa_cap": 0.10,
        "hairpin_size_guard_enable": True,
        "hairpin_size_guard_margin": 0.98,
        "hairpin_force_single_if_exceed": True,
        "hairpin_single_top_x": 33.0,
        "hairpin_single_top_y": 16.0,
        "hairpin_straight_in": 6.0,
        "hairpin_straight_out": 6.0,
        "hairpin_turn_angle_deg": 100.0,
        "hairpin_ref_speed": 0.95,
        "hairpin_speed_profile_gain": 30.0,
        "hairpin_speed_min": 0.55,
        "s_start_align": None,
    }


# 强制回头弯采用类直角双弯：避免旧内核里 TF12_PATH_CFG 残留 red_like
if bool(TF12_PATH_CFG.get("force_right_angle_pair", True)):
    TF12_PATH_CFG["hairpin_shape_mode"] = "right_angle_pair"

# 补齐类直角双弯参数默认值（兼容旧字典）
TF12_PATH_CFG.setdefault("hairpin_ra_right_x", 33.0)
TF12_PATH_CFG.setdefault("hairpin_ra_bottom_y", 2.5)
TF12_PATH_CFG.setdefault("hairpin_ra_top_y", 16.0)
TF12_PATH_CFG.setdefault("hairpin_ra_bottom_x0", 0.0)
TF12_PATH_CFG.setdefault("hairpin_ra_top_x0", 0.0)
TF12_PATH_CFG.setdefault("hairpin_ra_corner_r", 12.0)
TF12_PATH_CFG.setdefault("hairpin_ra_bottom_rise_len", 12.0)
TF12_PATH_CFG.setdefault("hairpin_ra_bottom_rise_h", 1.6)
TF12_PATH_CFG.setdefault("hairpin_ra_top_drop_len", 12.0)
TF12_PATH_CFG.setdefault("hairpin_ra_top_drop_h", 1.6)
TF12_PATH_CFG.setdefault("hairpin_ra_curvature_safe", True)
TF12_PATH_CFG.setdefault("hairpin_ra_side_wave_amp", 0.01)
TF12_PATH_CFG.setdefault("hairpin_ra_side_wave_harm", 1.0)
TF12_PATH_CFG.setdefault("hairpin_ra_top_wave_amp", 0.008)
TF12_PATH_CFG.setdefault("hairpin_ra_bottom_wave_amp", 0.008)
TF12_PATH_CFG.setdefault("hairpin_ra_smooth_win", 45)
TF12_PATH_CFG.setdefault("hairpin_ra_smooth_pass", 6)
TF12_PATH_CFG.setdefault("hairpin_kappa_cap", 0.10)
TF12_PATH_CFG.setdefault("hairpin_size_guard_enable", True)
TF12_PATH_CFG.setdefault("hairpin_size_guard_margin", 0.98)
TF12_PATH_CFG.setdefault("hairpin_force_single_if_exceed", True)
TF12_PATH_CFG.setdefault("hairpin_single_top_x", 33.0)
TF12_PATH_CFG.setdefault("hairpin_single_top_y", 16.0)

def _build_dlc_path(path_length, vx_ref, dt_val, enable_curved=True):
    T_traj = path_length / vx_ref
    t_arr = np.arange(0, T_traj + dt_val, dt_val)
    x_arr = vx_ref * t_arr
    if enable_curved:
        A1 = float(TF12_PATH_CFG.get("dlc_amp_1", 0.95))
        A2 = float(TF12_PATH_CFG.get("dlc_amp_2", -0.95))
        sigma_ratio = float(TF12_PATH_CFG.get("dlc_sigma_ratio", 0.14))
        x1 = 0.30 * path_length
        x2 = 0.72 * path_length
        sigma1 = sigma_ratio * path_length
        sigma2 = sigma_ratio * path_length
        y_dlc = A1 * np.exp(-0.5 * ((x_arr - x1) / sigma1) ** 2) + A2 * np.exp(-0.5 * ((x_arr - x2) / sigma2) ** 2)
        y_wave = 0.06 * np.sin(2.0 * np.pi * 2.0 * x_arr / path_length) * np.exp(-((x_arr - 0.5 * path_length) / (0.52 * path_length)) ** 2)
        y_arr = y_dlc + y_wave
    else:
        y_arr = np.zeros_like(x_arr)

    s_arr, psi_arr, kappa_arr = build_test_frenet_path_from_xy(x_arr, y_arr)
    return {
        "mode": "dlc",
        "path_length": float(path_length),
        "t_ref": t_arr,
        "traj_length": int(t_arr.size),
        "x_path": x_arr,
        "y_ref_path": y_arr,
        "s_ref_path": s_arr,
        "psi_ref_path": psi_arr,
        "curvature_ref_path": kappa_arr,
        "meta": {"kind": "double_lane_change"},
    }


def _compute_hairpin_radius(payload_cfg, mpc_cfg, path_cfg):
    payload_l = float(payload_cfg.get("payload_length", 5.0))
    payload_w = float(payload_cfg.get("payload_width", 2.0))
    veh_l = float(path_cfg.get("hairpin_vehicle_length", 4.4))
    veh_w = float(path_cfg.get("hairpin_vehicle_width", 1.9))
    clr = float(path_cfg.get("hairpin_clearance", 1.2))

    # 几何包络约束：货物+车辆组合体在弯道中不自交、不压边
    envelope_half_diag = 0.5 * np.hypot(payload_l + veh_l, payload_w + veh_w)
    r_geom_min = envelope_half_diag + clr

    # 控制可行性约束：根据前轮转角上限给出可跟踪半径下界（保守）
    wheelbase_eff = max(float(payload_l), 1.0)
    steer_lim = max(float(mpc_cfg.get("steer_limit", 0.12)), 1e-3)
    r_ctrl_min = wheelbase_eff / np.tan(steer_lim)

    sf = float(max(path_cfg.get("hairpin_radius_safety_factor", 1.20), 1.0))
    r_auto = max(r_geom_min, r_ctrl_min) * sf

    r_user = path_cfg.get("hairpin_radius_override", None)
    if r_user is not None:
        try:
            r_user = float(r_user)
            if np.isfinite(r_user) and r_user > 0.0:
                r_auto = max(r_auto, r_user)
        except Exception:
            pass

    kappa_geom_max = 1.0 / max(r_geom_min, 1e-6)
    kappa_ctrl_max = 1.0 / max(r_ctrl_min, 1e-6)
    kappa_limit = min(kappa_geom_max, kappa_ctrl_max)

    return float(r_auto), {
        "r_geom_min": float(r_geom_min),
        "r_ctrl_min": float(r_ctrl_min),
        "envelope_half_diag": float(envelope_half_diag),
        "radius_safety_factor": float(sf),
        "kappa_geom_max": float(kappa_geom_max),
        "kappa_ctrl_max": float(kappa_ctrl_max),
        "kappa_limit": float(kappa_limit),
    }


def _build_hairpin_path(v_ref, dt_val, payload_cfg, mpc_cfg, path_cfg):
    def _smooth_sig(v, win):
        win = int(max(1, win))
        if win <= 2:
            return v
        if win % 2 == 0:
            win += 1
        pad = win // 2
        ker = np.ones(win, dtype=float) / float(win)
        vp = np.pad(v, (pad, pad), mode="edge")
        return np.convolve(vp, ker, mode="valid")

    def _resample_xy(x_raw, y_raw, v_cmd, dt_cmd):
        ds_nom = max(v_cmd * dt_cmd, 0.04)
        seg = np.hypot(np.diff(x_raw), np.diff(y_raw))
        s_raw = np.concatenate([[0.0], np.cumsum(seg)])
        s_total = float(max(s_raw[-1], ds_nom))
        t_arr = np.arange(0.0, s_total / v_cmd + dt_cmd, dt_cmd)
        s_arr = np.clip(v_cmd * t_arr, 0.0, s_total)
        x_arr = np.interp(s_arr, s_raw, x_raw)
        y_arr = np.interp(s_arr, s_raw, y_raw)
        s_path, psi_path, kappa_path = build_test_frenet_path_from_xy(x_arr, y_arr)
        return t_arr, s_total, x_arr, y_arr, s_path, psi_path, kappa_path

    def _build_single_turn_hairpin(v_cmd, dt_cmd, cfg, meta_r):
        x0 = float(cfg.get("hairpin_ra_bottom_x0", 0.0))
        y0 = float(cfg.get("hairpin_ra_bottom_y", 2.5))
        xr = float(cfg.get("hairpin_ra_right_x", 40.0))
        r_corner = float(max(0.5, cfg.get("hairpin_ra_corner_r", 9.5)))
        rise_len = float(max(0.5, cfg.get("hairpin_ra_bottom_rise_len", 10.0)))
        rise_h = float(max(0.0, cfg.get("hairpin_ra_bottom_rise_h", 1.8)))
        side_wave_amp = float(max(0.0, cfg.get("hairpin_ra_side_wave_amp", 0.02)))
        side_wave_harm = float(max(0.5, cfg.get("hairpin_ra_side_wave_harm", 1.0)))
        smooth_win = int(max(1, cfg.get("hairpin_ra_smooth_win", 33)))
        smooth_pass = int(max(0, cfg.get("hairpin_ra_smooth_pass", 4)))

        top_x_cfg = cfg.get("hairpin_single_top_x", None)
        top_y_cfg = cfg.get("hairpin_single_top_y", None)
        y_top = float(cfg.get("hairpin_ra_top_y", 22.0))
        if top_y_cfg is not None:
            try:
                y_top = float(top_y_cfg)
            except Exception:
                pass
        y_top = max(y_top, y0 + 2.0 * r_corner + 1.0)

        x_vert = xr
        if top_x_cfg is not None:
            try:
                x_vert = float(top_x_cfg)
            except Exception:
                pass
        x_vert = max(x_vert, x0 + rise_len + r_corner + 5.0)

        ds_nom = max(v_cmd * dt_cmd, 0.04)

        def _nseg(L):
            return max(8, int(np.ceil(max(float(L), 1e-9) / ds_nom)) + 1)

        def _ease(u):
            return 0.5 - 0.5 * np.cos(np.pi * u)

        y_start = y0 - rise_h
        x1 = np.linspace(x0, x0 + rise_len, _nseg(rise_len))
        u1 = np.linspace(0.0, 1.0, x1.size)
        y1 = y_start + rise_h * _ease(u1)

        x2_start = float(x1[-1])
        x2_end = float(x_vert - r_corner)
        if x2_end <= x2_start + 0.5:
            x2_end = x2_start + 0.5
        x2 = np.linspace(x2_start, x2_end, _nseg(x2_end - x2_start))
        y2 = np.full_like(x2, y0)

        th3 = np.linspace(-0.5 * np.pi, 0.0, _nseg(0.5 * np.pi * r_corner))
        c3x = x_vert - r_corner
        c3y = y0 + r_corner
        x3 = c3x + r_corner * np.cos(th3)
        y3 = c3y + r_corner * np.sin(th3)

        y4_start = float(y3[-1])
        y4_end = float(y_top)
        if y4_end <= y4_start + 0.5:
            y4_end = y4_start + 0.5
        y4 = np.linspace(y4_start, y4_end, _nseg(y4_end - y4_start))
        u4 = np.linspace(0.0, 1.0, y4.size)
        x4 = np.full_like(y4, x_vert) + side_wave_amp * np.sin(np.pi * u4) * np.sin(side_wave_harm * np.pi * u4)

        x_raw = np.concatenate([x1, x2[1:], x3[1:], x4[1:]])
        y_raw = np.concatenate([y1, y2[1:], y3[1:], y4[1:]])

        x_s = x_raw.copy()
        y_s = y_raw.copy()
        for _ in range(smooth_pass):
            x_s = _smooth_sig(x_s, smooth_win)
            y_s = _smooth_sig(y_s, smooth_win)
        x_s[0], y_s[0] = x_raw[0], y_raw[0]
        x_s[-1], y_s[-1] = x_raw[-1], y_raw[-1]

        t_arr, s_total, x_arr, y_arr, s_path, psi_path, kappa_path = _resample_xy(x_s, y_s, v_cmd, dt_cmd)
        return {
            "mode": "hairpin",
            "path_length": float(s_total),
            "t_ref": t_arr,
            "traj_length": int(t_arr.size),
            "x_path": x_arr,
            "y_ref_path": y_arr,
            "s_ref_path": s_path,
            "psi_ref_path": psi_path,
            "curvature_ref_path": kappa_path,
            "meta": {
                "kind": "right_angle_single_hairpin",
                "shape_mode": "right_angle_single",
                "x_vert": float(x_vert),
                "y_bottom": float(y0),
                "y_top": float(y_top),
                "corner_r": float(r_corner),
                "rise_len": float(rise_len),
                "rise_h": float(rise_h),
                **meta_r,
            },
        }

    shape_mode = str(path_cfg.get("hairpin_shape_mode", "red_like")).lower().strip()

    if shape_mode in ("right_angle_single", "single_right_angle", "ra_single"):
        R_turn, meta_r = _compute_hairpin_radius(payload_cfg, mpc_cfg, path_cfg)
        return _build_single_turn_hairpin(v_ref, dt_val, path_cfg, {**meta_r, "R_turn_ref": float(R_turn)})

    if shape_mode in ("right_angle_pair", "two_right_angle", "blue_like", "ra_pair"):
        R_turn, meta_r = _compute_hairpin_radius(payload_cfg, mpc_cfg, path_cfg)

        x_right = float(path_cfg.get("hairpin_ra_right_x", 45.5))
        y_bottom = float(path_cfg.get("hairpin_ra_bottom_y", 2.5))
        y_top = float(path_cfg.get("hairpin_ra_top_y", 39.6))
        x_left_bottom = float(path_cfg.get("hairpin_ra_bottom_x0", 0.0))
        x_top_left = float(path_cfg.get("hairpin_ra_top_x0", 0.0))
        r_corner = float(max(0.5, path_cfg.get("hairpin_ra_corner_r", 7.0)))
        rise_len = float(max(0.5, path_cfg.get("hairpin_ra_bottom_rise_len", 12.0)))
        rise_h = float(max(0.0, path_cfg.get("hairpin_ra_bottom_rise_h", 2.3)))
        top_drop_len = float(max(0.5, path_cfg.get("hairpin_ra_top_drop_len", 12.0)))
        top_drop_h = float(max(0.0, path_cfg.get("hairpin_ra_top_drop_h", 2.4)))
        side_wave_amp = float(max(0.0, path_cfg.get("hairpin_ra_side_wave_amp", 0.08)))
        side_wave_harm = float(max(0.5, path_cfg.get("hairpin_ra_side_wave_harm", 1.0)))
        top_wave_amp = float(max(0.0, path_cfg.get("hairpin_ra_top_wave_amp", 0.06)))
        bottom_wave_amp = float(max(0.0, path_cfg.get("hairpin_ra_bottom_wave_amp", 0.05)))
        smooth_win = int(max(1, path_cfg.get("hairpin_ra_smooth_win", 19)))
        smooth_pass = int(max(0, path_cfg.get("hairpin_ra_smooth_pass", 2)))

        # 可选：按可跟踪最小半径放大角点圆角
        if bool(path_cfg.get("hairpin_ra_curvature_safe", False)):
            r_corner = max(r_corner, R_turn)

        # 几何保护，避免段长/高度退化
        y_top = max(y_top, y_bottom + 2.0 * r_corner + 2.0)
        x_right = max(x_right, x_left_bottom + rise_len + r_corner + 8.0)
        x_top_left = min(x_top_left, x_right - r_corner - 2.0)

        rise_len = min(rise_len, max(1.0, (x_right - r_corner) - x_left_bottom - 1.0))
        top_drop_len = min(top_drop_len, max(1.0, (x_right - r_corner) - x_top_left - 1.0))

        y_start = y_bottom - rise_h

        ds_nom = max(v_ref * dt_val, 0.04)

        def _nseg(L):
            return max(8, int(np.ceil(max(float(L), 1e-9) / ds_nom)) + 1)

        def _ease(u):
            return 0.5 - 0.5 * np.cos(np.pi * u)

        # 段1：底部左侧缓升
        x1 = np.linspace(x_left_bottom, x_left_bottom + rise_len, _nseg(rise_len))
        u1 = np.linspace(0.0, 1.0, x1.size)
        y1 = y_start + rise_h * _ease(u1)

        # 段2：底部直线
        x2_start = float(x1[-1])
        x2_end = float(x_right - r_corner)
        if x2_end <= x2_start + 0.5:
            x2_end = x2_start + 0.5
        x2 = np.linspace(x2_start, x2_end, _nseg(x2_end - x2_start))
        u2 = np.linspace(0.0, 1.0, x2.size)
        y2 = np.full_like(x2, y_bottom) + bottom_wave_amp * np.sin(np.pi * u2) ** 2

        # 段3：右下近直角弯（1/4圆）
        th3 = np.linspace(-0.5 * np.pi, 0.0, _nseg(0.5 * np.pi * r_corner))
        c3x = x_right - r_corner
        c3y = y_bottom + r_corner
        x3 = c3x + r_corner * np.cos(th3)
        y3 = c3y + r_corner * np.sin(th3)

        # 段4：右侧竖直段
        y4_start = float(y3[-1])
        y4_end = float(y_top - r_corner)
        if y4_end <= y4_start + 0.5:
            y4_end = y4_start + 0.5
        y4 = np.linspace(y4_start, y4_end, _nseg(y4_end - y4_start))
        u4 = np.linspace(0.0, 1.0, y4.size)
        x4 = np.full_like(y4, x_right) + side_wave_amp * np.sin(np.pi * u4) * np.sin(side_wave_harm * np.pi * u4)

        # 段5：右上近直角弯（1/4圆）
        th5 = np.linspace(0.0, 0.5 * np.pi, _nseg(0.5 * np.pi * r_corner))
        c5x = x_right - r_corner
        c5y = y_top - r_corner
        x5 = c5x + r_corner * np.cos(th5)
        y5 = c5y + r_corner * np.sin(th5)

        # 段6：顶部直线
        x6_start = float(x5[-1])
        x6_end = float(x_top_left + top_drop_len)
        if x6_end >= x6_start - 0.5:
            x6_end = x6_start - 0.5
        x6 = np.linspace(x6_start, x6_end, _nseg(abs(x6_start - x6_end)))
        u6 = np.linspace(0.0, 1.0, x6.size)
        y6 = np.full_like(x6, y_top) - top_wave_amp * np.sin(np.pi * u6) ** 2

        # 段7：顶部左侧缓降（与示意图蓝线一致）
        x7 = np.linspace(x6_end, x_top_left, _nseg(abs(x6_end - x_top_left)))
        u7 = np.linspace(0.0, 1.0, x7.size)
        y7 = y_top - top_drop_h * _ease(u7)

        x_raw = np.concatenate([x1, x2[1:], x3[1:], x4[1:], x5[1:], x6[1:], x7[1:]])
        y_raw = np.concatenate([y1, y2[1:], y3[1:], y4[1:], y5[1:], y6[1:], y7[1:]])

        x_s = x_raw.copy()
        y_s = y_raw.copy()
        for _ in range(smooth_pass):
            x_s = _smooth_sig(x_s, smooth_win)
            y_s = _smooth_sig(y_s, smooth_win)
        x_s[0], y_s[0] = x_raw[0], y_raw[0]
        x_s[-1], y_s[-1] = x_raw[-1], y_raw[-1]

        t_arr, s_total, x_arr, y_arr, s_path, psi_path, kappa_path = _resample_xy(x_s, y_s, v_ref, dt_val)
        kappa_cap = float(path_cfg.get("hairpin_kappa_cap", 0.22))
        if np.max(np.abs(kappa_path)) > kappa_cap:
            sw_auto = int(max(smooth_win, 33))
            for _ in range(4):
                x_arr = _smooth_sig(x_arr, sw_auto)
                y_arr = _smooth_sig(y_arr, sw_auto)
            s_path, psi_path, kappa_path = build_test_frenet_path_from_xy(x_arr, y_arr)

        if bool(path_cfg.get("hairpin_size_guard_enable", True)):
            kappa_limit = float(meta_r.get("kappa_limit", np.inf))
            margin = float(np.clip(path_cfg.get("hairpin_size_guard_margin", 0.98), 0.80, 1.10))
            kappa_peak = float(np.max(np.abs(kappa_path)))
            if np.isfinite(kappa_limit) and kappa_peak > margin * kappa_limit:
                print(
                    "[TF12][hairpin][WARN] curvature exceeds size/control limit: "
                    f"kappa_peak={kappa_peak:.4f}, limit={kappa_limit:.4f}. "
                    "fallback -> right_angle_single"
                )
                if bool(path_cfg.get("hairpin_force_single_if_exceed", True)):
                    cfg_single = dict(path_cfg)
                    cfg_single["hairpin_shape_mode"] = "right_angle_single"
                    return _build_hairpin_path(v_ref, dt_val, payload_cfg, mpc_cfg, cfg_single)

        return {
            "mode": "hairpin",
            "path_length": float(s_total),
            "t_ref": t_arr,
            "traj_length": int(t_arr.size),
            "x_path": x_arr,
            "y_ref_path": y_arr,
            "s_ref_path": s_path,
            "psi_ref_path": psi_path,
            "curvature_ref_path": kappa_path,
            "meta": {
                "kind": "right_angle_pair_hairpin",
                "shape_mode": shape_mode,
                "x_right": float(x_right),
                "y_bottom": float(y_bottom),
                "y_top": float(y_top),
                "x_left_bottom": float(x_left_bottom),
                "x_top_left": float(x_top_left),
                "corner_r": float(r_corner),
                "rise_len": float(rise_len),
                "rise_h": float(rise_h),
                "top_drop_len": float(top_drop_len),
                "top_drop_h": float(top_drop_h),
                "R_turn_ref": float(R_turn),
                "kappa_peak": float(np.max(np.abs(kappa_path))),
                **meta_r,
            },
        }

    if shape_mode in ("red_like", "red", "hook"):
        entry = float(max(0.0, path_cfg.get("hairpin_entry_straight", 1.0)))
        top = float(max(0.0, path_cfg.get("hairpin_top_straight", 8.0)))
        x_bulge = float(max(entry + 8.0, path_cfg.get("hairpin_red_bulge_x", 38.0)))
        y_top = float(max(10.0, path_cfg.get("hairpin_red_height", 40.0)))

        a = x_bulge - entry
        b = 0.5 * y_top

        # 可选：启用后会按可跟踪半径放大曲线（会偏离“红色示意”形状）
        R_turn, meta_r = _compute_hairpin_radius(payload_cfg, mpc_cfg, path_cfg)
        if bool(path_cfg.get("hairpin_red_curvature_safe", False)):
            b = max(b, np.sqrt(max(a * R_turn, 1e-6)))
            y_top = 2.0 * b

        ds_nom = max(v_ref * dt_val, 0.04)
        n1 = max(8, int(np.ceil(entry / ds_nom)) + 1)
        n2 = max(180, int(np.ceil(np.pi * (a + b) / ds_nom)))
        n3 = max(8, int(np.ceil(top / ds_nom)) + 1)

        # 底部短直线
        x1 = np.linspace(0.0, entry, n1)
        y1 = np.zeros_like(x1)

        # 红色示意的主回头段：x先增后减，y单调上升
        th = np.linspace(0.0, np.pi, n2)
        x2 = entry + a * np.sin(th)
        y2 = b * (1.0 - np.cos(th))

        # 顶部向左短直线
        x3 = np.linspace(entry, entry - top, n3)
        y3 = np.full_like(x3, 2.0 * b)

        x_raw = np.concatenate([x1, x2[1:], x3[1:]])
        y_raw = np.concatenate([y1, y2[1:], y3[1:]])

        seg = np.hypot(np.diff(x_raw), np.diff(y_raw))
        s_raw = np.concatenate([[0.0], np.cumsum(seg)])
        s_total = float(max(s_raw[-1], ds_nom))

        t_arr = np.arange(0.0, s_total / v_ref + dt_val, dt_val)
        s_arr = np.clip(v_ref * t_arr, 0.0, s_total)
        x_arr = np.interp(s_arr, s_raw, x_raw)
        y_arr = np.interp(s_arr, s_raw, y_raw)

        s_path, psi_path, kappa_path = build_test_frenet_path_from_xy(x_arr, y_arr)
        return {
            "mode": "hairpin",
            "path_length": float(s_total),
            "t_ref": t_arr,
            "traj_length": int(t_arr.size),
            "x_path": x_arr,
            "y_ref_path": y_arr,
            "s_ref_path": s_path,
            "psi_ref_path": psi_path,
            "curvature_ref_path": kappa_path,
            "meta": {
                "kind": "red_like_hairpin",
                "shape_mode": shape_mode,
                "entry": float(entry),
                "top": float(top),
                "x_bulge": float(x_bulge),
                "y_top": float(y_top),
                "R_turn_ref": float(R_turn),
                **meta_r,
            },
        }

    # 备用：保留原圆弧构造
    R_turn, meta_r = _compute_hairpin_radius(payload_cfg, mpc_cfg, path_cfg)
    L_in = float(path_cfg.get("hairpin_straight_in", 40.0))
    L_out = float(path_cfg.get("hairpin_straight_out", 40.0))
    theta_deg = float(np.clip(path_cfg.get("hairpin_turn_angle_deg", 100.0), 90.0, 170.0))
    theta = np.deg2rad(theta_deg)

    s1 = L_in
    s2 = s1 + R_turn * theta
    s_total = s2 + L_out

    t_arr = np.arange(0.0, s_total / v_ref + dt_val, dt_val)
    s_arr = v_ref * t_arr

    x_arr = np.zeros_like(s_arr)
    y_arr = np.zeros_like(s_arr)

    for i, s in enumerate(s_arr):
        if s <= s1:
            x_arr[i] = s
            y_arr[i] = 0.0
        elif s <= s2:
            th = (s - s1) / R_turn
            x_arr[i] = L_in + R_turn * np.sin(th)
            y_arr[i] = R_turn * (1.0 - np.cos(th))
        else:
            ds = s - s2
            x_end = L_in + R_turn * np.sin(theta)
            y_end = R_turn * (1.0 - np.cos(theta))
            hdg = theta
            x_arr[i] = x_end + ds * np.cos(hdg)
            y_arr[i] = y_end + ds * np.sin(hdg)

    s_path, psi_path, kappa_path = build_test_frenet_path_from_xy(x_arr, y_arr)
    return {
        "mode": "hairpin",
        "path_length": float(s_total),
        "t_ref": t_arr,
        "traj_length": int(t_arr.size),
        "x_path": x_arr,
        "y_ref_path": y_arr,
        "s_ref_path": s_path,
        "psi_ref_path": psi_path,
        "curvature_ref_path": kappa_path,
        "meta": {
            "kind": "u_turn_hairpin",
            "R_turn": float(R_turn),
            "turn_angle_deg": float(theta_deg),
            "straight_in": float(L_in),
            "straight_out": float(L_out),
            **meta_r,
        },
    }


# Build path library
TF12_PATH_LIBRARY = {}
TF12_PATH_LIBRARY["dlc"] = _build_dlc_path(
    path_length=float(MPC_CFG["path_length"]),
    vx_ref=vx_nom,
    dt_val=dt,
    enable_curved=bool(RUN_CFG["enable_curved_tracking"]),
)

if bool(TF12_PATH_CFG.get("build_hairpin", True)):
    v_h = float(np.clip(TF12_PATH_CFG.get("hairpin_ref_speed", 1.8), 1.0, vx_nom))
    TF12_PATH_LIBRARY["hairpin"] = _build_hairpin_path(
        v_ref=v_h,
        dt_val=dt,
        payload_cfg=A1_PAYLOAD_CFG,
        mpc_cfg=MPC_CFG,
        path_cfg=TF12_PATH_CFG,
    )

active_mode = str(TF12_PATH_CFG.get("active_mode", "dlc")).lower().strip()
if active_mode not in TF12_PATH_LIBRARY:
    raise ValueError(f"Unknown TF12_PATH_CFG['active_mode']={active_mode}. available={list(TF12_PATH_LIBRARY.keys())}")

path_pack = TF12_PATH_LIBRARY[active_mode]
path_length = float(path_pack["path_length"])
t_ref = path_pack["t_ref"]
traj_length = int(path_pack["traj_length"])
x_path = path_pack["x_path"]
y_ref_path = path_pack["y_ref_path"]
s_ref_path = path_pack["s_ref_path"]
psi_ref_path = path_pack["psi_ref_path"]
curvature_ref_path = path_pack["curvature_ref_path"]

kappa_abs = np.abs(curvature_ref_path)
if active_mode == "hairpin":
    v_cap = float(np.clip(TF12_PATH_CFG.get("hairpin_ref_speed", 1.8), 1.0, vx_nom))
    curv_gain = float(TF12_PATH_CFG.get("hairpin_speed_profile_gain", 32.0))
    v_min = float(TF12_PATH_CFG.get("hairpin_speed_min", 0.9))
else:
    v_cap = float(np.clip(TF12_PATH_CFG.get("dlc_speed_cap", vx_nom), 1.0, vx_nom))
    curv_gain = float(MPC_CFG["speed_profile_curv_gain"])
    v_min = float(MPC_CFG["speed_profile_min"])

vx_ref_profile = v_cap / (1.0 + curv_gain * kappa_abs)
vx_ref_profile = np.clip(vx_ref_profile, v_min, v_cap)

smooth_beta = float(np.clip(MPC_CFG["speed_profile_smooth"], 0.0, 0.98))
for ii in range(1, vx_ref_profile.size):
    vx_ref_profile[ii] = smooth_beta * vx_ref_profile[ii - 1] + (1.0 - smooth_beta) * vx_ref_profile[ii]

slow_start = float(MPC_CFG["terminal_slowdown_start_s"])
slow_floor = float(np.clip(MPC_CFG["terminal_slowdown_floor"], 0.4, 1.0))
if s_ref_path[-1] > slow_start:
    slow_ratio = np.clip((s_ref_path - slow_start) / max(s_ref_path[-1] - slow_start, 1e-6), 0.0, 1.0)
    terminal_scale = 1.0 - (1.0 - slow_floor) * slow_ratio
    vx_ref_profile = np.clip(vx_ref_profile * terminal_scale, max(0.75, 0.85 * v_min), v_cap)

x_ref_raw = np.zeros((num_states, traj_length))
x_ref_raw[0, :] = s_ref_path
x_ref_raw[1, :] = 0.0
x_ref_raw[2, :] = 0.0
x_ref_raw[3, :] = vx_ref_profile
x_ref_raw[4, :] = 0.0
x_ref_raw[5, :] = vx_ref_profile * curvature_ref_path

# 纵向对齐：默认把参考 s 起点对齐到当前载荷中心起点，降低 e_s 初始偏置
s_start_align_cfg = TF12_PATH_CFG.get("s_start_align", None)
if s_start_align_cfg is None:
    s_start_align = float(x0_payload_center[0])
else:
    s_start_align = float(s_start_align_cfg)
if abs(s_start_align) > 1e-12:
    s_ref_path = s_ref_path + s_start_align
    x_ref_raw[0, :] = x_ref_raw[0, :] + s_start_align
    print(f"[TF12][path] applied s_start_align = {s_start_align:.3f} m")

x_ref_scaled = standardizer_x_kdnn.transform(x_ref_raw.T).T

print(f"[TF12][path] active_mode={active_mode}, traj_length={traj_length}, s_end={s_ref_path[-1]:.2f} m, v_cap={v_cap:.2f}")
if active_mode == "hairpin":
    print("[TF12][path] hairpin meta:", path_pack["meta"])

if bool(TF12_PATH_CFG.get("plot_path_library", True)):
    plt.figure(figsize=(10, 4.8))
    for mode, pack in TF12_PATH_LIBRARY.items():
        lw = 2.5 if mode == active_mode else 1.7
        alpha = 0.95 if mode == active_mode else 0.65
        plt.plot(pack["x_path"], pack["y_ref_path"], linewidth=lw, alpha=alpha, label=f"{mode} ({pack['meta']['kind']})")
    plt.title("TF12 路径库（DLC误差优化 + 短回头弯）")
    plt.xlabel("x [m]")
    plt.ylabel("y [m]")
    plt.grid(True, alpha=0.30)
    plt.axis("equal")
    plt.legend()
    plt.tight_layout()
    plt.show()




In [ ]:
# =========================
# Solver settings（防卡死调优 + 可开关）
# =========================
solver_settings = {}
solver_settings["gen_embedded_ctrl"] = False
solver_settings["warm_start"] = True
solver_settings["polish"] = False
solver_settings["polish_refine_iter"] = 3
solver_settings["scaling"] = True
solver_settings["adaptive_rho"] = True
solver_settings["check_termination"] = 20
solver_settings["max_iter"] = 1400
solver_settings["eps_abs"] = 2.0e-3
solver_settings["eps_rel"] = 2.0e-3
solver_settings["eps_prim_inf"] = 1e-3
solver_settings["eps_dual_inf"] = 1e-3
solver_settings["sqp_step_size"] = 0.85
solver_settings["du_clip"] = 0.45
solver_settings["retry_on_fail"] = 1
solver_settings["retry_relax_factor"] = 1.8
solver_settings["accept_solved_inaccurate"] = True
solver_settings["accept_partial_solution"] = True
solver_settings["accept_partial_prim_res"] = 1.2e-1
solver_settings["accept_partial_dual_res"] = 2.0e-1
solver_settings["verbose"] = False

if RUN_CFG["enable_solver_guard"]:
    solver_settings["time_limit"] = MPC_CFG["osqp_time_limit"]

# 初始化参考轨迹列表
x_ref_mpc_vehicles = []



In [ ]:
# =========================
# MPC horizon / ref preview + A1 rigid-payload reference bundle
# =========================
N_lin_noadapt = MPC_CFG["horizon"]
max_iter_lin = MPC_CFG["max_sqp_iters"]

A1_COORDINATOR = None
A1_REF_BUNDLE = None
A1_COORDINATOR, A1_REF_BUNDLE = tf12_runtime.build_a1_reference_bundle(
    payload_module=payload_a1,
    payload_cfg=A1_PAYLOAD_CFG,
    standardizer_x=standardizer_x_kdnn,
    x_ref_raw=x_ref_raw,
    leader_idx=FORMATION_CFG["leader_index"],
    horizon_pad=N_lin_noadapt + 2,
)
A1_REFERENCE_VERSION = A1_NOTEBOOK_VERSION

x_ref_raw_vehicles = A1_REF_BUNDLE["raw_vehicle_refs"]
a1_ref_vehicle_histories = A1_REF_BUNDLE["corner_ref_histories"]
x_ref_mpc_vehicles = A1_REF_BUNDLE["mpc_vehicle_refs"]

print("[A1] rigid reference bundle ready")
print("[A1] reference version:", A1_REFERENCE_VERSION)
print("[A1] ref_team_hist shape:", A1_REF_BUNDLE["team_ref_hist"].shape)
print("[A1] ref_vehicle_histories[0] shape:", a1_ref_vehicle_histories[0].shape)
print("[A1] x_ref_mpc_vehicles[0] shape:", x_ref_mpc_vehicles[0].shape)

# =========================
# Relaxed raw bounds -> scaled bounds
# =========================
xmin_raw = np.array([-10.0, -5.0, -1.2, 0.5, -4.0, -3.0])
xmax_raw = np.array([500.0, 5.0, 1.2, 8.0, 4.0, 3.0])

xmin_lin_noadapt = standardizer_x_kdnn.transform(xmin_raw.reshape(1, -1)).flatten()
xmax_lin_noadapt = standardizer_x_kdnn.transform(xmax_raw.reshape(1, -1)).flatten()

umax_lin_noadapt = np.array([MPC_CFG["steer_limit"], MPC_CFG["accel_limit"]], dtype=float)
umin_lin_noadapt = -umax_lin_noadapt

# ====================== 基础权重 ======================
Q_base_lin = scipy.sparse.diags([
    0.0,
    3500.0,
    1200.0,
    8.0,
    1.0,
    1.0
])

QN_base_lin = scipy.sparse.diags([
    0.0,
    5000.0,
    2000.0,
    12.0,
    1.0,
    1.0
])

R_mpc_lin_noadapt = scipy.sparse.diags([120.0, 12.0])



from control_files.tf12.core_utils import (
    spectral_project_matrix,
    configure_raw_linear_fit_context,
    fit_raw_linear_model_scaled,
)

configure_raw_linear_fit_context(
    num_states=num_states,
    num_inputs=num_inputs,
    standardizer_x=standardizer_x_kdnn,
    standardizer_u=standardizer_u_kdnn,
)

# 无 Koopman 对照模型（直接在 scaled 原状态空间拟合线性模型）
A_raw_scaled, B_raw_scaled = fit_raw_linear_model_scaled(xs_train, us_train, ridge_lambda=8e-5)
A_raw_scaled_stable = spectral_project_matrix(A_raw_scaled, radius=0.998)
C_raw_scaled = np.eye(num_states)

print("[对照模型] A_raw_scaled shape:", A_raw_scaled.shape, "| B_raw_scaled shape:", B_raw_scaled.shape)
print("[对照模型] spectral radius (stable proj):", np.max(np.abs(np.linalg.eigvals(A_raw_scaled_stable))))


In [ ]:

# =========================
# TF12 控制/自适应 helper 绑定（模块化）
# =========================
leader_idx = FORMATION_CFG["leader_index"]
formation_targets = []
for v in range(num_vehicles):
    formation_targets.append({
        "ds": float(x0_vehicles[v][0] - x0_vehicles[leader_idx][0]),
        "dey": float(x0_vehicles[v][1] - x0_vehicles[leader_idx][1])
    })

import importlib
import control_files.tf12.mpc_helpers as tf12_mpc_helpers

tf12_mpc_helpers = importlib.reload(tf12_mpc_helpers)
tf12_mpc_helpers.bind_context(
    MPC_CFG=MPC_CFG,
    PPC_CFG=PPC_CFG,
    FORMATION_CFG=FORMATION_CFG,
    KOOPMAN_CFG=KOOPMAN_CFG,
    A_lin=A_lin,
    A_lin_stable=A_lin_stable,
    B_lin=B_lin,
    C_lin=C_lin,
    model_koop_dnn_lin=model_koop_dnn_lin,
    xs_train=xs_train,
    us_train=us_train,
    spectral_project_matrix=spectral_project_matrix,
    num_states=num_states,
    AdaptNet_linear=AdaptNet_linear,
)

adaptive_mpc_weights = tf12_mpc_helpers.adaptive_mpc_weights
maybe_update_ppc = tf12_mpc_helpers.maybe_update_ppc
coop_gain_scale_by_s = tf12_mpc_helpers.coop_gain_scale_by_s
apply_coop_correction = tf12_mpc_helpers.apply_coop_correction
apply_emergency_guard = tf12_mpc_helpers.apply_emergency_guard
enforce_progress_and_stability = tf12_mpc_helpers.enforce_progress_and_stability

should_take_adapt_sample = tf12_mpc_helpers.should_take_adapt_sample
clip_fro_norm = tf12_mpc_helpers.clip_fro_norm
fit_koopman_bilinear_matrices = tf12_mpc_helpers.fit_koopman_bilinear_matrices
get_tf9_model_pack = tf12_mpc_helpers.get_tf9_model_pack
resolve_adapt_mode = tf12_mpc_helpers.resolve_adapt_mode
weighted_stack = tf12_mpc_helpers.weighted_stack

fit_delta_linear_ridge = tf12_mpc_helpers.fit_delta_linear_ridge
fit_delta_linear_net = tf12_mpc_helpers.fit_delta_linear_net
fit_delta_bilinear_ridge = tf12_mpc_helpers.fit_delta_bilinear_ridge
apply_online_delta = tf12_mpc_helpers.apply_online_delta

# 仅保留兼容名；A1 主流程使用 tf12_runtime.run_tf12_main
run_tf9_main = tf12_mpc_helpers.run_tf9_main_placeholder

print("[A1] 控制/自适应 helper 已模块化导入: control_files/tf12/mpc_helpers.py")


In [ ]:

# =========================
# TF12 指标函数与状态判定（模块化）
# =========================
import importlib
import control_files.tf12.metrics_utils as tf12_metrics

tf12_metrics = importlib.reload(tf12_metrics)

_is_solver_success_status = tf12_metrics.is_solver_success_status
_solver_success_rate = tf12_metrics.solver_success_rate
_full_path_rate = tf12_metrics.full_path_rate
_build_case_metrics = lambda result: tf12_metrics.build_case_metrics(result, num_vehicles=num_vehicles)


In [ ]:

# =========================
# TF12 case runner helper（模块化）
# =========================
import importlib
import control_files.tf12.case_runner_utils as tf12_case_runner

tf12_case_runner = importlib.reload(tf12_case_runner)

run_tf10_case = tf12_case_runner.make_run_tf10_case(
    tf10_modules_default=TF10_MODULES_DEFAULT,
    run_cfg=RUN_CFG,
    method_cfg=METHOD_CFG,
    mpc_cfg=MPC_CFG,
    solver_settings=solver_settings,
    run_case_fn=run_tf9_main,
    build_case_metrics_fn=_build_case_metrics,
    namespace=globals(),
)

print_tf10_case_summary = tf12_case_runner.print_tf10_case_summary


In [ ]:

# =========================
# TF12 explicit runtime context（模块化）
# =========================
from control_files.tf12.context_utils import build_a1_runtime_context_from_globals

A1_RUNTIME_KEYS = ['ADAPT_CFG', 'METHOD_CFG', 'MPC_CFG', 'N_lin_noadapt', 'NonlinearMPCController', 'PPC_CFG', 'QN_base_lin', 'Q_base_lin', 'RUN_CFG', 'R_mpc_lin_noadapt', 'SNR_DB', 'TF11_MODULES_MAIN', '_build_case_metrics', 'adaptive_mpc_weights', 'apply_coop_correction', 'apply_emergency_guard', 'apply_online_delta', 'clip_closed_loop_state', 'curvature_ref_path', 'dt', 'enforce_progress_and_stability', 'fit_delta_bilinear_ridge', 'fit_delta_linear_net', 'fit_delta_linear_ridge', 'get_tf9_model_pack', 'is_finite_vector', 'leader_idx', 'lift_scaled', 'max_iter_lin', 'maybe_update_ppc', 'model_koop_dnn_lin', 'net_params_lin', 'num_inputs', 'num_states', 'num_vehicles', 'resolve_adapt_mode', 's_ref_path', 'safe_FK_step', 'scale_state', 'should_take_adapt_sample', 'solver_settings', 'standardizer_x_kdnn', 'sys_pars', 'sys_pars_new', 'traj_length', 'umax_lin_noadapt', 'umin_lin_noadapt', 'vx_nom', 'weighted_stack', 'x0_vehicles', 'x_ref_raw', 'xmax_lin_noadapt', 'xmin_lin_noadapt']

build_a1_runtime_context = lambda: build_a1_runtime_context_from_globals(
    runtime_keys=A1_RUNTIME_KEYS,
    global_ns=globals(),
    coordinator=A1_COORDINATOR,
    ref_bundle=A1_REF_BUNDLE,
    notebook_version=A1_NOTEBOOK_VERSION,
)

print("[A1] explicit runtime context helper ready. key_count =", len(A1_RUNTIME_KEYS))


In [ ]:

# =========================
# TF12 main method: rigid payload + bilinear adaptive Koopman
# （在线自适应更新调试打印：X/Y 形状 + ||ΔA||, ||ΔB||）
# =========================
payload_a1 = importlib.reload(payload_a1)
tf12_runtime = importlib.reload(tf12_runtime)

A1_COORDINATOR, A1_REF_BUNDLE = tf12_runtime.build_a1_reference_bundle(
    payload_module=payload_a1,
    payload_cfg=A1_PAYLOAD_CFG,
    standardizer_x=standardizer_x_kdnn,
    x_ref_raw=x_ref_raw,
    leader_idx=FORMATION_CFG["leader_index"],
    horizon_pad=N_lin_noadapt + 2,
)
A1_REFERENCE_VERSION = A1_NOTEBOOK_VERSION
x_ref_raw_vehicles = A1_REF_BUNDLE["raw_vehicle_refs"]
a1_ref_vehicle_histories = A1_REF_BUNDLE["corner_ref_histories"]
x_ref_mpc_vehicles = A1_REF_BUNDLE["mpc_vehicle_refs"]
A1_RUNTIME_CTX = build_a1_runtime_context()

# -------------------------
# 在线自适应更新调试包装（只在主程序 cell 生效）
# -------------------------
_adapt_dbg_state = {
    "idx": 0,
    "mode": None,
    "x_shape": None,
    "y_shape": None,
}

_orig_fit_linear_net = A1_RUNTIME_CTX["fit_delta_linear_net"]
_orig_fit_linear_ridge = A1_RUNTIME_CTX["fit_delta_linear_ridge"]
_orig_fit_bilinear_ridge = A1_RUNTIME_CTX["fit_delta_bilinear_ridge"]
_orig_apply_online_delta = A1_RUNTIME_CTX["apply_online_delta"]


def _dbg_fit_linear_net(Z_hist, U_hist, dZ_hist, adapt_cfg, nz, nu, net_params, del_A_prev=None, del_B_prev=None):
    X_dbg = np.hstack([Z_hist, U_hist])
    Y_dbg = dZ_hist
    _adapt_dbg_state["mode"] = "linear_net"
    _adapt_dbg_state["x_shape"] = tuple(X_dbg.shape)
    _adapt_dbg_state["y_shape"] = tuple(Y_dbg.shape)
    return _orig_fit_linear_net(Z_hist, U_hist, dZ_hist, adapt_cfg, nz, nu, net_params, del_A_prev=del_A_prev, del_B_prev=del_B_prev)


def _dbg_fit_linear_ridge(Z_hist, U_hist, dZ_hist, W, ridge_lambda=1e-4):
    X_dbg = np.hstack([Z_hist, U_hist])
    Y_dbg = dZ_hist
    _adapt_dbg_state["mode"] = "linear_ridge"
    _adapt_dbg_state["x_shape"] = tuple(X_dbg.shape)
    _adapt_dbg_state["y_shape"] = tuple(Y_dbg.shape)
    return _orig_fit_linear_ridge(Z_hist, U_hist, dZ_hist, W, ridge_lambda=ridge_lambda)


def _dbg_fit_bilinear_ridge(Z_hist, U_hist, dZ_hist, W, ridge_lambda=1e-4):
    ZU = np.array([np.kron(Z_hist[i], U_hist[i]) for i in range(Z_hist.shape[0])], dtype=np.float64)
    X_dbg = np.hstack([Z_hist, ZU])
    Y_dbg = dZ_hist
    _adapt_dbg_state["mode"] = "bilinear_ridge"
    _adapt_dbg_state["x_shape"] = tuple(X_dbg.shape)
    _adapt_dbg_state["y_shape"] = tuple(Y_dbg.shape)
    return _orig_fit_bilinear_ridge(Z_hist, U_hist, dZ_hist, W, ridge_lambda=ridge_lambda)


def _dbg_apply_online_delta(dynamics_obj, dA, dB, structure_name, adapt_cfg):
    _adapt_dbg_state["idx"] += 1
    raw_da = float(np.linalg.norm(dA))
    raw_db = float(np.linalg.norm(dB))
    da_norm, db_norm = _orig_apply_online_delta(dynamics_obj, dA, dB, structure_name, adapt_cfg)

    print(
        f"[ADAPT-UPDATE {_adapt_dbg_state['idx']:03d}] "
        f"mode={_adapt_dbg_state.get('mode', 'unknown')} | "
        f"X={_adapt_dbg_state.get('x_shape')} | Y={_adapt_dbg_state.get('y_shape')} | "
        f"||ΔA||={raw_da:.4e} (applied {da_norm:.4e}) | "
        f"||ΔB||={raw_db:.4e} (applied {db_norm:.4e})"
    )
    return da_norm, db_norm


A1_RUNTIME_CTX["fit_delta_linear_net"] = _dbg_fit_linear_net
A1_RUNTIME_CTX["fit_delta_linear_ridge"] = _dbg_fit_linear_ridge
A1_RUNTIME_CTX["fit_delta_bilinear_ridge"] = _dbg_fit_bilinear_ridge
A1_RUNTIME_CTX["apply_online_delta"] = _dbg_apply_online_delta

print()
print("=" * 90)
print("TF12 main method: rigid rectangular payload + bilinear adaptive Koopman")
print("=" * 90)
print("[A1] notebook version:", A1_NOTEBOOK_VERSION)
print("[A1] reference version:", A1_REFERENCE_VERSION)
print("[A1] runtime context keys:", len(A1_RUNTIME_CTX))
print("[A1] online adaptation debug print: ON (X/Y shape + ||ΔA||, ||ΔB||)")

main_result = tf12_runtime.run_tf12_main(A1_RUNTIME_CTX, payload_a1, A1_PAYLOAD_CFG, A1_MAIN_CHANGE_MASK)
compare_results = {"tf12_main": main_result}
print_tf10_case_summary("tf12_main", main_result)
print("Payload force summary:", main_result["payload_force_summary"])
print("Payload variation mask:", main_result["payload_change_mask"])
print("Connection summary:", main_result.get("connection_summary", {}))


In [ ]:
# =========================
# TF12 main result refill
# =========================
if main_result is None:
    raise RuntimeError("main_result is None. Please run the TF12 main cell first.")
xt_actual_vehicles = main_result["xt_actual_vehicles"]
u_vehicles = main_result["u_vehicles"]
z_vehicles = main_result["z_vehicles"]
fail_counts = main_result["fail_counts"]
solver_status_hist = main_result["solver_status_hist"]
terminated_early = main_result["terminated_early"]
team_state_hist = main_result["team_state_hist"]
team_input_hist = main_result["team_input_hist"]
payload_force_hist = main_result["payload_force_hist"]
payload_force_summary = main_result["payload_force_summary"]
payload_change_mask = main_result["payload_change_mask"]
ref_team_hist = main_result["ref_team_hist"]
ref_vehicle_histories = main_result["ref_vehicle_histories"]
ref_leader_relative_targets = main_result["ref_leader_relative_targets"]
actual_sim_steps = int(main_result["sim_steps"])
begin_time = main_result["begin_time"]
end_linear_noadapt = main_result["end_time"]

print("[A1] refill source: main_result only")
print("[A1] actual_sim_steps =", actual_sim_steps)
print("[A1] payload variation mask =", payload_change_mask)


In [ ]:

# =========================
# TF12 可开关可视化（静态图 + 动画 + 输入图）
# =========================
import warnings
warnings.filterwarnings("ignore", message="Glyph .* missing from font", category=UserWarning)

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

_TF11_A1_VIZ_DEFAULT = {
    # 路径图
    "enable_static_main_path": True,
    "enable_static_subplots": True,

    # 动画（最耗时，默认关）
    "enable_realtime_animation": False,
    "animation_embed_limit_mb": 40.0,
    "max_embed_frames": 420,
    "force_frame_step": None,

    # 输入图
    "enable_input_plots": True,
    "save_input_figures": True,
    "input_fig_dpi": 150,
}

_tf12_viz_cfg_existing = globals().get('TF11_A1_VIZ_CFG', None)
if not isinstance(_tf12_viz_cfg_existing, dict):
    TF11_A1_VIZ_CFG = dict(_TF11_A1_VIZ_DEFAULT)
else:
    _tmp_cfg = dict(_TF11_A1_VIZ_DEFAULT)
    _tmp_cfg.update(_tf12_viz_cfg_existing)
    TF11_A1_VIZ_CFG = _tmp_cfg

print('[A1][viz] config =', TF11_A1_VIZ_CFG)

styles = [
    {'color': 'tab:blue', 'linestyle': '--', 'marker': 'o', 'linewidth': 1.8, 'label': '车1 (前左)'},
    {'color': 'tab:purple', 'linestyle': '-.', 'marker': 's', 'linewidth': 1.8, 'label': '车2 (前右)'},
    {'color': 'tab:green', 'linestyle': (0, (5, 2, 1, 2)), 'marker': '^', 'linewidth': 1.8, 'label': '车3 (后左)'},
    {'color': 'tab:red', 'linestyle': (0, (1, 1)), 'marker': 'D', 'linewidth': 1.8, 'label': '车4 (后右)'}
]

# =========================
# A) 静态总图
# =========================
if TF11_A1_VIZ_CFG['enable_static_main_path']:
    plt.figure(figsize=(12, 8))
    plt.plot(x_path, y_ref_path, label="货物中心参考路径", linewidth=2.5, color='black')

    for v in range(num_vehicles):
        n_hist = min(actual_sim_steps + 1, xt_actual_vehicles[v].shape[0], ref_vehicle_histories[v].shape[0])
        if n_hist <= 1:
            print(f"[WARN] 车{v + 1} 有效长度不足，无轨迹可画")
            continue

        s_ref_v = np.clip(ref_vehicle_histories[v][:n_hist, 0], s_ref_path.min() - 10.0, s_ref_path.max() + 10.0)
        ey_ref_v = ref_vehicle_histories[v][:n_hist, 1]
        x_ref_v, y_ref_v = frenet_to_global(s_ref_v, ey_ref_v, s_ref_path, x_path, y_ref_path, psi_ref_path)

        s_v = np.clip(xt_actual_vehicles[v][:n_hist, 0], s_ref_path.min() - 10.0, s_ref_path.max() + 10.0)
        ey_v = xt_actual_vehicles[v][:n_hist, 1]
        x_v, y_v = frenet_to_global(s_v, ey_v, s_ref_path, x_path, y_ref_path, psi_ref_path)

        plt.plot(
            x_ref_v, y_ref_v,
            color=styles[v]['color'], linestyle=':', linewidth=1.0, alpha=0.65,
            label=f"{styles[v]['label']} 参考"
        )
        plt.plot(
            x_v, y_v,
            color=styles[v]['color'], linestyle=styles[v]['linestyle'], linewidth=styles[v]['linewidth'],
            label=styles[v]['label']
        )

    plt.xlabel("x [m]")
    plt.ylabel("y [m]")
    plt.title("TF12 四车刚性搬运路径跟踪")
    plt.legend(loc='upper right', ncol=2)
    plt.grid(True, alpha=0.6)
    plt.tight_layout()
    plt.show()
else:
    print('[A1][viz] 跳过静态总图（enable_static_main_path=False）')

# =========================
# B) 动画（开关）
# =========================
if TF11_A1_VIZ_CFG['enable_realtime_animation']:
    from IPython.display import HTML, display
    import matplotlib as mpl
    import importlib
    import tf12_realtime_viz as _tf12_viz
    _tf12_viz = importlib.reload(_tf12_viz)

    mpl.rcParams['animation.embed_limit'] = max(
        float(mpl.rcParams.get('animation.embed_limit', 20.0)),
        float(TF11_A1_VIZ_CFG.get('animation_embed_limit_mb', 40.0))
    )

    anim_steps = int(main_result.get('sim_steps_cap', actual_sim_steps))
    run_steps = int(main_result.get('sim_steps', actual_sim_steps))
    if run_steps < anim_steps:
        print(f"[INFO] 主仿真提前结束: run_steps={run_steps}, sim_steps_cap={anim_steps}, full_path_reached={main_result.get('full_path_reached', False)}")
        print('[INFO] 动画将播放到 sim_steps_cap；提前结束后的时段会保持车辆末状态。')

    force_step = TF11_A1_VIZ_CFG.get('force_frame_step', None)
    if force_step is not None:
        frame_step = max(1, int(force_step))
    else:
        max_embed_frames = int(max(120, TF11_A1_VIZ_CFG.get('max_embed_frames', 420)))
        frame_step = max(1, int(np.ceil(anim_steps / max_embed_frames)))

    if frame_step > 1:
        print(f"[INFO] 动画抽帧启用: frame_step={frame_step} (总步数={anim_steps})")

    try:
        fig_anim, anim = _tf12_viz.animate_a1_tracking_rectangles(
            xt_actual_vehicles=xt_actual_vehicles,
            ref_vehicle_histories=ref_vehicle_histories,
            actual_sim_steps=anim_steps,
            s_ref_path=s_ref_path,
            x_path=x_path,
            y_ref_path=y_ref_path,
            psi_ref_path=psi_ref_path,
            frenet_to_global=frenet_to_global,
            dt=dt,
            vehicle_length=4.4,
            vehicle_width=1.9,
            tail_points=max(60, int(140 / frame_step)),
            frame_step=frame_step,
            interval_ms=max(20, int(1000 * dt * frame_step)),
            figsize=(10.5, 5.2),
            dark_theme=True,
            show_vehicle_refs=True,
        )
        fps_show = max(8, int(round(1.0 / max(dt * frame_step, 1e-6))))
        display(HTML(anim.to_jshtml(fps=fps_show)))
        plt.close(fig_anim)
    except Exception as e:
        print(f"[WARN] 实时动画生成失败: {e}")
else:
    print('[A1][viz] 跳过实时动画（enable_realtime_animation=False）')

# =========================
# C) 轨迹诊断（轻量，始终保留）
# =========================
print("=== 各车轨迹诊断 ===")
for v in range(num_vehicles):
    n_hist = min(actual_sim_steps + 1, xt_actual_vehicles[v].shape[0], ref_vehicle_histories[v].shape[0])
    if n_hist > 0:
        final_s = xt_actual_vehicles[v][n_hist - 1, 0]
        ref_final_s = ref_vehicle_histories[v][n_hist - 1, 0]
        print(f"车{v + 1} → 有效步数: {n_hist} | 最终s位置: {final_s:.2f} m | 参考终点s: {ref_final_s:.2f} m")
    else:
        print(f"车{v + 1} → 无有效轨迹")

# =========================
# D) 四子图路径（开关）
# =========================
if TF11_A1_VIZ_CFG['enable_static_subplots']:
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
    plt.rcParams['axes.unicode_minus'] = False

    styles = [
        {'color': 'tab:blue', 'linestyle': '--', 'marker': 'o', 'markevery': 30, 'linewidth': 1.8, 'label': '车1 (前左)'},
        {'color': 'tab:purple', 'linestyle': '-.', 'marker': 's', 'markevery': 30, 'linewidth': 1.8, 'label': '车2 (前右)'},
        {'color': 'tab:green', 'linestyle': (0, (5, 2, 1, 2)), 'marker': '^', 'markevery': 30, 'linewidth': 1.8, 'label': '车3 (后左)'},
        {'color': 'tab:red', 'linestyle': (0, (1, 1)), 'marker': 'D', 'markevery': 30, 'linewidth': 1.8, 'label': '车4 (后右)'}
    ]

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()

    for v in range(num_vehicles):
        ax = axes[v]
        ax.plot(x_path, y_ref_path, label="货物中心参考", linewidth=2.5, color='black', linestyle='-')

        n_hist = min(actual_sim_steps + 1, xt_actual_vehicles[v].shape[0], ref_vehicle_histories[v].shape[0])
        if n_hist <= 1:
            ax.set_title(f"车{v + 1} - 轨迹无效")
            continue

        s_ref_v = np.clip(ref_vehicle_histories[v][:n_hist, 0], s_ref_path.min() - 10.0, s_ref_path.max() + 10.0)
        ey_ref_v = ref_vehicle_histories[v][:n_hist, 1]
        x_ref_v, y_ref_v = frenet_to_global(s_ref_v, ey_ref_v, s_ref_path, x_path, y_ref_path, psi_ref_path)

        s_v = np.clip(xt_actual_vehicles[v][:n_hist, 0], s_ref_path.min() - 10.0, s_ref_path.max() + 10.0)
        ey_v = xt_actual_vehicles[v][:n_hist, 1]
        x_v, y_v = frenet_to_global(s_v, ey_v, s_ref_path, x_path, y_ref_path, psi_ref_path)

        ax.plot(x_ref_v, y_ref_v, color=styles[v]['color'], linestyle=':', linewidth=1.0, alpha=0.65, label=f"{styles[v]['label']} 参考")
        ax.plot(
            x_v, y_v,
            color=styles[v]['color'], linestyle=styles[v]['linestyle'], marker=styles[v]['marker'],
            markevery=styles[v]['markevery'], linewidth=styles[v]['linewidth'], label=styles[v]['label']
        )

        ax.set_title(f"{styles[v]['label']} 路径跟踪")
        ax.set_xlabel("x [m]")
        ax.set_ylabel("y [m]")
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.6)

    fig.suptitle("TF12 四车刚性搬运路径跟踪", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('[A1][viz] 跳过四子图路径（enable_static_subplots=False）')

# =========================
# E) 输入量图（开关）
# =========================
if TF11_A1_VIZ_CFG['enable_input_plots']:
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
    plt.rcParams['axes.unicode_minus'] = False

    N_ctrl = u_vehicles[0].shape[0]
    t_ctrl = np.arange(N_ctrl) * dt

    if curvature_ref_path.shape[0] == 0:
        curvature_ff = np.zeros(N_ctrl)
    elif curvature_ref_path.shape[0] >= N_ctrl:
        curvature_ff = curvature_ref_path[:N_ctrl]
    else:
        curvature_ff = np.pad(curvature_ref_path, (0, N_ctrl - curvature_ref_path.shape[0]), mode='edge')

    delta_ff_ref_shared = 0.8 * (sys_pars["lf"] + sys_pars["lr"]) * curvature_ff

    colors_v = ['tab:blue', 'tab:purple', 'tab:green', 'tab:red']
    labels_v = ['车1 (前左)', '车2 (前右)', '车3 (后左)', '车4 (后右)']
    ls_v = ['--', '-.', (0, (5, 2, 1, 2)), (0, (1, 1))]

    fig1, axes1 = plt.subplots(nrows=num_vehicles, ncols=2, figsize=(16, 3.2 * num_vehicles), sharex=True)
    fig1.suptitle('四车控制输入量（子图分开）', fontsize=15, fontweight='bold')

    for v in range(num_vehicles):
        delta_v = u_vehicles[v][:, 0]
        ax_v = u_vehicles[v][:, 1]

        ax_left = axes1[v, 0]
        ax_left.plot(t_ctrl, np.rad2deg(delta_v), color=colors_v[v], linestyle=ls_v[v], linewidth=1.5, label=labels_v[v])
        ax_left.axhline(0, color='gray', linewidth=0.8, linestyle=':')
        ax_left.plot(t_ctrl, np.rad2deg(delta_ff_ref_shared), color='gray', linewidth=1.0, linestyle='--', alpha=0.6, label='前馈参考')
        ax_left.set_ylabel(labels_v[v] + "\n$\\delta$ (deg)", fontsize=9)
        ax_left.legend(fontsize=8, loc='upper right')
        ax_left.grid(True, alpha=0.3)
        if v == 0:
            ax_left.set_title('前轮转角 $\\delta$', fontsize=11)
        if v == num_vehicles - 1:
            ax_left.set_xlabel('时间 (s)', fontsize=10)

        ax_right = axes1[v, 1]
        ax_right.plot(t_ctrl, ax_v, color=colors_v[v], linestyle=ls_v[v], linewidth=1.5, label=labels_v[v])
        ax_right.axhline(0, color='gray', linewidth=0.8, linestyle=':')
        ax_right.set_ylabel(labels_v[v] + "\n$a_x$ (m/s$^2$)", fontsize=9)
        ax_right.legend(fontsize=8, loc='upper right')
        ax_right.grid(True, alpha=0.3)
        if v == 0:
            ax_right.set_title('纵向加速度 $a_x$', fontsize=11)
        if v == num_vehicles - 1:
            ax_right.set_xlabel('时间 (s)', fontsize=10)

    plt.tight_layout(rect=[0, 0.02, 1, 0.98])
    if TF11_A1_VIZ_CFG['save_input_figures']:
        plt.savefig('inputs_separated.png', dpi=int(TF11_A1_VIZ_CFG.get('input_fig_dpi', 150)), bbox_inches='tight')
        print('[OK] 子图分开版已保存：inputs_separated.png')
    plt.show()

    fig2, (ax_top, ax_bot) = plt.subplots(nrows=2, ncols=1, figsize=(14, 8), sharex=True)
    fig2.suptitle('四车控制输入量（合图）', fontsize=15, fontweight='bold')

    ax_top.plot(t_ctrl, np.rad2deg(delta_ff_ref_shared), color='gray', linewidth=1.4, linestyle='--', alpha=0.8, label='前馈参考 $\\delta_{ff}$')
    for v in range(num_vehicles):
        delta_v = u_vehicles[v][:, 0]
        ax_top.plot(t_ctrl, np.rad2deg(delta_v), color=colors_v[v], linestyle=ls_v[v], linewidth=1.8, label=labels_v[v])
    ax_top.axhline(0, color='gray', linewidth=0.8, linestyle=':')
    ax_top.set_ylabel('前轮转角 $\\delta$ (deg)', fontsize=11)
    ax_top.grid(True, alpha=0.3)
    ax_top.legend(fontsize=9, ncol=3, loc='upper right')

    for v in range(num_vehicles):
        ax_v = u_vehicles[v][:, 1]
        ax_bot.plot(t_ctrl, ax_v, color=colors_v[v], linestyle=ls_v[v], linewidth=1.8, label=labels_v[v])
    ax_bot.axhline(0, color='gray', linewidth=0.8, linestyle=':')
    ax_bot.set_ylabel('纵向加速度 $a_x$ (m/s$^2$)', fontsize=11)
    ax_bot.set_xlabel('时间 (s)', fontsize=11)
    ax_bot.grid(True, alpha=0.3)
    ax_bot.legend(fontsize=9, ncol=4, loc='upper right')

    plt.tight_layout(rect=[0, 0.02, 1, 0.98])
    plt.show()

    fig3, axes3 = plt.subplots(nrows=2, ncols=2, figsize=(16, 10), sharex=True)
    fig3.suptitle('四车控制输入量（各车双轴子图）', fontsize=15, fontweight='bold')
    axes3 = axes3.flatten()

    for v in range(num_vehicles):
        ax_main = axes3[v]
        delta_v = np.rad2deg(u_vehicles[v][:, 0])
        ax_v = u_vehicles[v][:, 1]

        ax_main.plot(t_ctrl, delta_v, color=colors_v[v], linestyle='-', linewidth=1.8, label='$\\delta$ (deg)')
        ax_main.axhline(0, color='gray', linewidth=0.8, linestyle=':')
        ax_main.set_ylabel('$\\delta$ (deg)', color=colors_v[v], fontsize=10)
        ax_main.tick_params(axis='y', labelcolor=colors_v[v])
        ax_main.grid(True, alpha=0.25)

        ax_twin = ax_main.twinx()
        ax_twin.plot(t_ctrl, ax_v, color=colors_v[v], linestyle=':', linewidth=1.8, alpha=0.9, label='$a_x$ (m/s$^2$)')
        ax_twin.set_ylabel('$a_x$ (m/s$^2$)', color=colors_v[v], fontsize=10)
        ax_twin.tick_params(axis='y', labelcolor=colors_v[v])

        ax_main.set_title(labels_v[v], fontsize=12, fontweight='bold')
        ax_main.set_xlabel('时间 (s)', fontsize=10)

        h1, l1 = ax_main.get_legend_handles_labels()
        h2, l2 = ax_twin.get_legend_handles_labels()
        ax_main.legend(h1 + h2, l1 + l2, fontsize=8, loc='upper right')

    plt.tight_layout(rect=[0, 0.02, 1, 0.97])
    if TF11_A1_VIZ_CFG['save_input_figures']:
        plt.savefig('inputs_dual_axis.png', dpi=int(TF11_A1_VIZ_CFG.get('input_fig_dpi', 150)), bbox_inches='tight')
        print('[OK] 双轴子图版已保存：inputs_dual_axis.png')
    plt.show()
else:
    print('[A1][viz] 跳过输入图（enable_input_plots=False）')


In [ ]:
# =========================
# TF12 横向误差与纵向误差
# =========================
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle('横向误差与纵向误差', fontsize=14, fontweight='bold')

colors_v = ['tab:blue', 'tab:purple', 'tab:green', 'tab:red']
labels_v = ['车1', '车2', '车3', '车4']
err_stats = []

for v in range(num_vehicles):
    x_hist_full = xt_actual_vehicles[v]
    ref_hist_full = ref_vehicle_histories[v]
    n_hist = min(actual_sim_steps + 1, x_hist_full.shape[0], ref_hist_full.shape[0])
    if n_hist <= 1:
        print(f"车{v+1}: 无有效误差数据")
        continue

    x_hist = x_hist_full[:n_hist, :]
    ref_hist = ref_hist_full[:n_hist, :]
    t_eval = np.arange(n_hist) * dt

    e_long = x_hist[:, 0] - ref_hist[:, 0]
    e_lat = x_hist[:, 1] - ref_hist[:, 1]

    valid = np.isfinite(e_long) & np.isfinite(e_lat)
    if not np.any(valid):
        print(f"车{v+1}: 无有效误差数据")
        continue

    c = colors_v[v % len(colors_v)]
    label = labels_v[v] if v < len(labels_v) else f'车{v+1}'

    ax1.plot(t_eval[valid], e_lat[valid], color=c, linewidth=1.8, label=label)
    ax2.plot(t_eval[valid], e_long[valid], color=c, linewidth=1.8, label=label)

    rmse_lat = float(np.sqrt(np.mean(e_lat[valid] ** 2)))
    rmse_long = float(np.sqrt(np.mean(e_long[valid] ** 2)))
    max_lat = float(np.max(np.abs(e_lat[valid])))
    max_long = float(np.max(np.abs(e_long[valid])))
    err_stats.append((v + 1, rmse_lat, max_lat, rmse_long, max_long))

ax1.axhline(0.0, color='black', linestyle=':', linewidth=1.0)
ax2.axhline(0.0, color='black', linestyle=':', linewidth=1.0)
ax1.set_ylabel('横向误差 e_y [m]')
ax2.set_ylabel('纵向误差 e_s [m]')
ax2.set_xlabel('时间 [s]')
ax1.grid(True, alpha=0.35)
ax2.grid(True, alpha=0.35)
ax1.legend(loc='upper right', ncol=min(2, num_vehicles))
ax2.legend(loc='upper right', ncol=min(2, num_vehicles))

plt.tight_layout()
plt.show()

print("=== 误差统计（相对于刚体四角参考）===")
for vid, rmse_lat, max_lat, rmse_long, max_long in err_stats:
    print(
        f"车{vid}: "
        f"RMSE(e_y)={rmse_lat:.4f} m, Max|e_y|={max_lat:.4f} m | "
        f"RMSE(e_s)={rmse_long:.4f} m, Max|e_s|={max_long:.4f} m"
    )


In [ ]:
# =========================
# TF12 相图分析（Phase Portrait）
# =========================
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

if 'xt_actual_vehicles' not in globals():
    raise RuntimeError('xt_actual_vehicles 不存在，请先运行 TF12 主方法与结果回填单元。')

sim_steps_eval = int(main_result.get('sim_steps', xt_actual_vehicles[0].shape[0] - 1))
labels_v = ['车1 (前左)', '车2 (前右)', '车3 (后左)', '车4 (后右)']
colors = ['tab:blue', 'tab:purple', 'tab:green', 'tab:red']

# 1) e_y - e_psi 相图
fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
fig.suptitle('TF12 相图A：横向误差 e_y 与航向误差 e_psi', fontsize=14, fontweight='bold')

for v in range(num_vehicles):
    ax = axes[v // 2, v % 2]
    x_hist = xt_actual_vehicles[v][:sim_steps_eval + 1, :]
    ey = x_hist[:, 1]
    epsi = x_hist[:, 2]

    ax.plot(ey, epsi, color=colors[v], linewidth=1.6, label=labels_v[v])
    ax.scatter(ey[0], epsi[0], color='lime', s=36, marker='o', label='起点')
    ax.scatter(ey[-1], epsi[-1], color='red', s=36, marker='x', label='终点')
    ax.set_xlabel('e_y [m]')
    ax.set_ylabel('e_psi [rad]')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc='best')

plt.show()

# 2) v_y - r 相图（侧向速度-横摆角速度）
if xt_actual_vehicles[0].shape[1] >= 6:
    fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
    fig.suptitle('TF12 相图B：侧向速度 v_y 与横摆角速度 r', fontsize=14, fontweight='bold')

    for v in range(num_vehicles):
        ax = axes[v // 2, v % 2]
        x_hist = xt_actual_vehicles[v][:sim_steps_eval + 1, :]
        vy = x_hist[:, 4]
        r = x_hist[:, 5]

        ax.plot(vy, r, color=colors[v], linewidth=1.6, label=labels_v[v])
        ax.scatter(vy[0], r[0], color='lime', s=36, marker='o', label='起点')
        ax.scatter(vy[-1], r[-1], color='red', s=36, marker='x', label='终点')
        ax.set_xlabel('v_y [m/s]')
        ax.set_ylabel('r [rad/s]')
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8, loc='best')

    plt.show()
else:
    print('[TF12 相图B] 状态维度 < 6，跳过 v_y-r 相图。')




In [ ]:
# =========================
# TF12 相图C：横摆角速度 r 与横摆角加速度 r_dot
# =========================
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

if 'main_result' not in globals() or main_result is None:
    raise RuntimeError('main_result 不存在，请先运行 TF12 主方法单元。')
if 'xt_actual_vehicles' not in globals() or len(xt_actual_vehicles) == 0:
    raise RuntimeError('xt_actual_vehicles 不存在，请先运行结果回填单元。')
if 'dt' not in globals():
    raise RuntimeError('dt 不存在，请先运行基础配置与建模单元。')

sim_steps_phase = int(main_result.get('sim_steps', xt_actual_vehicles[0].shape[0] - 1))
sim_steps_phase = max(1, sim_steps_phase)


from control_files.tf12.core_utils import calc_r_and_rdot as _calc_r_and_rdot


# 1) 系统整体相平面（团队状态）
if 'team_state_hist' in globals() and isinstance(team_state_hist, np.ndarray) and team_state_hist.shape[1] >= 6:
    n_team = min(sim_steps_phase + 1, team_state_hist.shape[0])
    r_team, rdot_team = _calc_r_and_rdot(team_state_hist[:n_team, 5], dt)
else:
    r_team = np.array([])
    rdot_team = np.array([])

r_ref = np.array([])
rdot_ref = np.array([])
if 'ref_team_hist' in globals() and isinstance(ref_team_hist, np.ndarray) and ref_team_hist.shape[1] >= 6:
    n_ref = min(sim_steps_phase + 1, ref_team_hist.shape[0])
    r_ref, rdot_ref = _calc_r_and_rdot(ref_team_hist[:n_ref, 5], dt)

fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)
if r_team.size > 0:
    ax.plot(r_team, rdot_team, color='tab:blue', linewidth=1.8, label='系统整体（实际）')
    ax.scatter(r_team[0], rdot_team[0], color='lime', s=40, marker='o', label='起点')
    ax.scatter(r_team[-1], rdot_team[-1], color='red', s=40, marker='x', label='终点')
if r_ref.size > 0:
    ax.plot(r_ref, rdot_ref, color='white', linestyle='--', linewidth=1.4, alpha=0.8, label='系统整体（参考）')

ax.axhline(0.0, color='gray', linestyle=':', linewidth=0.9)
ax.axvline(0.0, color='gray', linestyle=':', linewidth=0.9)
ax.set_xlabel('横摆角速度 r [rad/s]')
ax.set_ylabel('横摆角加速度 r_dot [rad/s²]')
ax.set_title('TF12 相图C1：系统整体 r - r_dot')
ax.grid(True, alpha=0.3)
ax.legend(loc='best', fontsize=9)
plt.show()

# 2) 各车辆相平面（四子图）
labels_v = ['车1 (前左)', '车2 (前右)', '车3 (后左)', '车4 (后右)']
colors_v = ['tab:blue', 'tab:purple', 'tab:green', 'tab:red']

veh_curves = []
for v in range(num_vehicles):
    x_hist = np.asarray(xt_actual_vehicles[v], dtype=float)
    n_hist = min(sim_steps_phase + 1, x_hist.shape[0])
    if n_hist <= 1 or x_hist.shape[1] < 6:
        veh_curves.append((np.array([]), np.array([])))
        continue
    r_v, rdot_v = _calc_r_and_rdot(x_hist[:n_hist, 5], dt)
    veh_curves.append((r_v, rdot_v))

# 统一轴范围，方便对比
r_all = np.concatenate([c[0] for c in veh_curves if c[0].size > 0], axis=0) if any(c[0].size > 0 for c in veh_curves) else np.array([0.0])
rdot_all = np.concatenate([c[1] for c in veh_curves if c[1].size > 0], axis=0) if any(c[1].size > 0 for c in veh_curves) else np.array([0.0])
r_lim = max(0.2, float(np.max(np.abs(r_all))) * 1.15)
rdot_lim = max(0.5, float(np.max(np.abs(rdot_all))) * 1.15)

fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
fig.suptitle('TF12 相图C2：各车辆 r - r_dot', fontsize=14, fontweight='bold')

for v in range(num_vehicles):
    ax = axes[v // 2, v % 2]
    r_v, rdot_v = veh_curves[v]
    if r_v.size <= 1:
        ax.set_title(f'{labels_v[v]}（数据不足）')
        ax.grid(True, alpha=0.3)
        continue

    ax.plot(r_v, rdot_v, color=colors_v[v], linewidth=1.7, label=labels_v[v])
    ax.scatter(r_v[0], rdot_v[0], color='lime', s=34, marker='o', label='起点')
    ax.scatter(r_v[-1], rdot_v[-1], color='red', s=34, marker='x', label='终点')

    ax.axhline(0.0, color='gray', linestyle=':', linewidth=0.8)
    ax.axvline(0.0, color='gray', linestyle=':', linewidth=0.8)
    ax.set_xlim(-r_lim, r_lim)
    ax.set_ylim(-rdot_lim, rdot_lim)
    ax.set_xlabel('r [rad/s]')
    ax.set_ylabel('r_dot [rad/s²]')
    ax.set_title(labels_v[v])
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best', fontsize=8)

plt.show()

print('=== r-r_dot 相图统计 ===')
if r_team.size > 0:
    print(f'系统整体: max|r|={np.max(np.abs(r_team)):.4f}, max|r_dot|={np.max(np.abs(rdot_team)):.4f}')
for v in range(num_vehicles):
    r_v, rdot_v = veh_curves[v]
    if r_v.size > 0:
        print(f'车{v+1}: max|r|={np.max(np.abs(r_v)):.4f}, max|r_dot|={np.max(np.abs(rdot_v)):.4f}')



In [ ]:

# =========================
# TF12 相轨迹簇 + 吸引域相图（系统整体 + 四车）
# 相平面：r 与 r_dot
# =========================
from scipy.spatial import cKDTree

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

if 'main_result' not in globals() or main_result is None:
    raise RuntimeError('main_result 不存在，请先运行 TF12 主方法单元。')
if 'team_state_hist' not in globals() or 'xt_actual_vehicles' not in globals():
    raise RuntimeError('team_state_hist / xt_actual_vehicles 缺失，请先运行结果回填单元。')
if 'dt' not in globals():
    raise RuntimeError('dt 不存在，请先运行基础配置单元。')

PHASE_BASIN_CFG = {
    # 吸引域底图
    'grid_size': 45,
    'sim_steps': 70,
    'k_neigh': 14,
    'dt_sim': float(dt),
    'r_bound_scale': 1.8,
    'rdot_bound_scale': 1.8,
    'attractor_tol_scale': 0.22,

    # 轨迹簇图（你要的“全是相轨迹”）
    'traj_only_samples': 120,
    'traj_only_len': 90,
    'traj_overlay_samples': 16,

    # 预处理
    'smooth_win': 7,
    'seed': 2026,
}


def _moving_average(arr, win):
    arr = np.asarray(arr, dtype=float).reshape(-1)
    if win <= 1 or arr.size < 3:
        return arr
    win = int(max(1, min(win, max(3, arr.size // 4))))
    if win % 2 == 0:
        win += 1
    ker = np.ones(win, dtype=float) / float(win)
    return np.convolve(arr, ker, mode='same')


def _phase_from_hist(r_hist, dt_val, smooth_win):
    r_hist = np.asarray(r_hist, dtype=float).reshape(-1)
    valid = np.isfinite(r_hist)
    r = r_hist[valid]
    if r.size < 8:
        return None
    r = _moving_average(r, smooth_win)
    rdot = np.gradient(r, float(dt_val))
    rdot = _moving_average(rdot, max(3, smooth_win - 2))
    rddot = np.gradient(rdot, float(dt_val))
    phase = np.column_stack([r, rdot])
    return phase, rddot


def _build_phase_model(phase_pts, rddot_targets):
    tree = cKDTree(phase_pts)
    y = np.asarray(rddot_targets, dtype=float).reshape(-1)

    def pred_rddot(x_query, k_neigh=12):
        xq = np.asarray(x_query, dtype=float)
        if xq.ndim == 1:
            xq = xq.reshape(1, 2)
        k_eff = int(max(1, min(k_neigh, phase_pts.shape[0])))
        d, idx = tree.query(xq, k=k_eff)
        if k_eff == 1:
            d = d.reshape(-1, 1)
            idx = idx.reshape(-1, 1)
        w = 1.0 / np.maximum(d, 1e-6)
        y_hat = np.sum(w * y[idx], axis=1) / np.sum(w, axis=1)
        return y_hat

    return pred_rddot


def _simulate_cloud(r0_grid, rd0_grid, pred_fn, cfg, attractor, r_lim, rd_lim):
    states = np.column_stack([r0_grid.reshape(-1), rd0_grid.reshape(-1)])
    alive = np.ones(states.shape[0], dtype=bool)

    for _ in range(int(cfg['sim_steps'])):
        if not np.any(alive):
            break
        s_alive = states[alive]
        rddot_hat = pred_fn(s_alive, k_neigh=cfg['k_neigh'])
        s_alive[:, 0] = s_alive[:, 0] + cfg['dt_sim'] * s_alive[:, 1]
        s_alive[:, 1] = s_alive[:, 1] + cfg['dt_sim'] * rddot_hat
        states[alive] = s_alive

        bad = (
            (~np.isfinite(states[:, 0])) | (~np.isfinite(states[:, 1])) |
            (np.abs(states[:, 0]) > r_lim) | (np.abs(states[:, 1]) > rd_lim)
        )
        alive = alive & (~bad)

    dist_final = np.sqrt((states[:, 0] - attractor[0]) ** 2 + (states[:, 1] - attractor[1]) ** 2)
    attract_tol = cfg['attractor_tol_scale'] * np.sqrt(r_lim ** 2 + rd_lim ** 2)
    stable = alive & (dist_final <= attract_tol)
    return stable.reshape(r0_grid.shape), states


def _simulate_traj(x0, pred_fn, cfg, r_lim, rd_lim, n_steps=60):
    x = np.asarray(x0, dtype=float).reshape(2)
    out = [x.copy()]
    for _ in range(int(n_steps)):
        if (not np.all(np.isfinite(x))) or (abs(x[0]) > r_lim) or (abs(x[1]) > rd_lim):
            break
        rdd = float(pred_fn(x, k_neigh=cfg['k_neigh'])[0])
        x = np.array([
            x[0] + cfg['dt_sim'] * x[1],
            x[1] + cfg['dt_sim'] * rdd,
        ], dtype=float)
        out.append(x.copy())
    return np.array(out, dtype=float)


sim_steps_phase = int(main_result.get('sim_steps', team_state_hist.shape[0] - 1))
sim_steps_phase = max(10, sim_steps_phase)

phase_items = []

# 系统整体
if isinstance(team_state_hist, np.ndarray) and team_state_hist.shape[1] >= 6:
    n_team = min(sim_steps_phase + 1, team_state_hist.shape[0])
    pack = _phase_from_hist(team_state_hist[:n_team, 5], dt, PHASE_BASIN_CFG['smooth_win'])
    if pack is not None:
        phase_pts, rddot_t = pack
        phase_items.append({
            'name': '系统整体',
            'color': 'tab:blue',
            'phase': phase_pts,
            'rddot': rddot_t,
        })

# 四车
labels_v = ['车1 (前左)', '车2 (前右)', '车3 (后左)', '车4 (后右)']
colors_v = ['tab:blue', 'tab:purple', 'tab:green', 'tab:red']
for v in range(num_vehicles):
    x_hist = np.asarray(xt_actual_vehicles[v], dtype=float)
    if x_hist.ndim != 2 or x_hist.shape[1] < 6:
        continue
    n_hist = min(sim_steps_phase + 1, x_hist.shape[0])
    pack = _phase_from_hist(x_hist[:n_hist, 5], dt, PHASE_BASIN_CFG['smooth_win'])
    if pack is None:
        continue
    phase_pts, rddot_t = pack
    phase_items.append({
        'name': labels_v[v] if v < len(labels_v) else f'车{v+1}',
        'color': colors_v[v % len(colors_v)],
        'phase': phase_pts,
        'rddot': rddot_t,
    })

if len(phase_items) == 0:
    raise RuntimeError('无可用相平面数据，无法绘制相轨迹/吸引域图。')

# 统一范围
r_all = np.concatenate([it['phase'][:, 0] for it in phase_items], axis=0)
rd_all = np.concatenate([it['phase'][:, 1] for it in phase_items], axis=0)
r_span = np.percentile(np.abs(r_all), 99)
rd_span = np.percentile(np.abs(rd_all), 99)
r_lim = max(0.20, float(PHASE_BASIN_CFG['r_bound_scale']) * float(r_span))
rd_lim = max(0.40, float(PHASE_BASIN_CFG['rdot_bound_scale']) * float(rd_span))

r_grid = np.linspace(-r_lim, r_lim, int(PHASE_BASIN_CFG['grid_size']))
rd_grid = np.linspace(-rd_lim, rd_lim, int(PHASE_BASIN_CFG['grid_size']))
RR, RRD = np.meshgrid(r_grid, rd_grid)

rng = np.random.default_rng(int(PHASE_BASIN_CFG['seed']))

# 先预计算每个对象的模型/稳定图
phase_models = []
for item in phase_items[:5]:
    phase_pts = item['phase']
    rddot_t = item['rddot']
    tail_n = max(8, int(0.12 * phase_pts.shape[0]))
    attractor = np.mean(phase_pts[-tail_n:, :], axis=0)
    pred_fn = _build_phase_model(phase_pts, rddot_t)
    stable_map, _ = _simulate_cloud(RR, RRD, pred_fn, PHASE_BASIN_CFG, attractor, r_lim, rd_lim)
    phase_models.append({
        **item,
        'pred_fn': pred_fn,
        'attractor': attractor,
        'stable_map': stable_map,
        'stable_ratio': float(np.mean(stable_map)),
    })

# =========================
# 图1：全是相轨迹（你要的风格）
# =========================
figA, axesA = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)
axesA = axesA.flatten()
figA.suptitle('TF12 相轨迹簇图（系统整体 + 四车）\n相平面: r - r_dot', fontsize=15, fontweight='bold')

for idx, model in enumerate(phase_models):
    ax = axesA[idx]
    phase_pts = model['phase']
    pred_fn = model['pred_fn']

    # 大量初值轨迹
    n_traj = int(PHASE_BASIN_CFG['traj_only_samples'])
    seed_idx = rng.integers(0, RR.size, size=n_traj)
    starts = np.column_stack([RR.reshape(-1)[seed_idx], RRD.reshape(-1)[seed_idx]])
    cmap = plt.cm.turbo
    for j, s0 in enumerate(starts):
        tr = _simulate_traj(s0, pred_fn, PHASE_BASIN_CFG, r_lim, rd_lim, n_steps=PHASE_BASIN_CFG['traj_only_len'])
        if tr.shape[0] > 1:
            ax.plot(tr[:, 0], tr[:, 1], color=cmap(j / max(1, n_traj - 1)), alpha=0.72, linewidth=0.9)

    # 历史轨迹和关键点
    ax.plot(phase_pts[:, 0], phase_pts[:, 1], color='white', linewidth=2.0, alpha=0.95, label='历史轨迹')
    ax.scatter(phase_pts[0, 0], phase_pts[0, 1], s=34, c='yellow', edgecolors='k', linewidths=0.6, marker='o', label='历史起点')
    ax.scatter(phase_pts[-1, 0], phase_pts[-1, 1], s=34, c='red', edgecolors='k', linewidths=0.6, marker='X', label='历史终点')
    ax.scatter(model['attractor'][0], model['attractor'][1], s=56, c='lime', edgecolors='k', linewidths=0.8, marker='*', label='吸引点')

    ax.set_title(f"{model['name']} | 相轨迹簇", fontsize=11)
    ax.set_xlabel('r [rad/s]')
    ax.set_ylabel('r_dot [rad/s$^2$]')
    ax.set_xlim(-r_lim, r_lim)
    ax.set_ylim(-rd_lim, rd_lim)
    ax.axhline(0.0, color='gray', linestyle=':', linewidth=0.8)
    ax.axvline(0.0, color='gray', linestyle=':', linewidth=0.8)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8, loc='upper right')

if len(axesA) > 5:
    ax_info = axesA[5]
    ax_info.axis('off')
    txt = (
        '相轨迹簇图说明:\n'
        '1) 每条彩色线对应一个不同初值\n'
        '2) 白线为本次主实验历史相轨迹\n'
        '3) 黄/红/绿分别是起点/终点/吸引点\n\n'
        f"traj_only_samples={PHASE_BASIN_CFG['traj_only_samples']}\n"
        f"traj_only_len={PHASE_BASIN_CFG['traj_only_len']}"
    )
    ax_info.text(0.05, 0.95, txt, va='top', ha='left', fontsize=11)

plt.show()

# =========================
# 图2：吸引域稳定区 + 相轨迹
# =========================
figB, axesB = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)
axesB = axesB.flatten()
figB.suptitle('TF12 吸引域相图（系统整体 + 四车）\n相平面: r - r_dot', fontsize=15, fontweight='bold')

for idx, model in enumerate(phase_models):
    ax = axesB[idx]
    phase_pts = model['phase']

    bg = np.where(model['stable_map'], 1.0, 0.0)
    ax.contourf(RR, RRD, bg, levels=[-0.1, 0.5, 1.1], alpha=0.26, colors=['#f59e9e', '#95d5b2'])
    ax.contour(RR, RRD, bg, levels=[0.5], colors='white', linewidths=1.0, alpha=0.9)

    # 少量轨迹叠加
    seed_idx = rng.integers(0, RR.size, size=int(PHASE_BASIN_CFG['traj_overlay_samples']))
    starts = np.column_stack([RR.reshape(-1)[seed_idx], RRD.reshape(-1)[seed_idx]])
    for s0 in starts:
        tr = _simulate_traj(s0, model['pred_fn'], PHASE_BASIN_CFG, r_lim, rd_lim, n_steps=PHASE_BASIN_CFG['traj_only_len'])
        if tr.shape[0] > 1:
            ax.plot(tr[:, 0], tr[:, 1], color='white', alpha=0.35, linewidth=0.9)

    ax.plot(phase_pts[:, 0], phase_pts[:, 1], color=model['color'], linewidth=2.0, label='历史轨迹')
    ax.scatter(phase_pts[0, 0], phase_pts[0, 1], s=35, c='yellow', edgecolors='k', linewidths=0.6, marker='o', label='历史起点')
    ax.scatter(phase_pts[-1, 0], phase_pts[-1, 1], s=35, c='red', edgecolors='k', linewidths=0.6, marker='X', label='历史终点')
    ax.scatter(model['attractor'][0], model['attractor'][1], s=56, c='lime', edgecolors='k', linewidths=0.8, marker='*', label='吸引点')

    ax.set_title(f"{model['name']} | 稳定占比={model['stable_ratio']*100:.1f}%", fontsize=11)
    ax.set_xlabel('r [rad/s]')
    ax.set_ylabel('r_dot [rad/s$^2$]')
    ax.set_xlim(-r_lim, r_lim)
    ax.set_ylim(-rd_lim, rd_lim)
    ax.axhline(0.0, color='gray', linestyle=':', linewidth=0.8)
    ax.axvline(0.0, color='gray', linestyle=':', linewidth=0.8)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8, loc='upper right')

if len(axesB) > 5:
    ax_info = axesB[5]
    ax_info.axis('off')
    txt = (
        '稳定区判定规则:\n'
        '1) 相轨迹在仿真步内不越界、不发散\n'
        '2) 最终状态落入吸引点邻域\n\n'
        f"grid_size={PHASE_BASIN_CFG['grid_size']}\n"
        f"sim_steps={PHASE_BASIN_CFG['sim_steps']}\n"
        f"k_neigh={PHASE_BASIN_CFG['k_neigh']}\n"
        f"traj_overlay_samples={PHASE_BASIN_CFG['traj_overlay_samples']}"
    )
    ax_info.text(0.05, 0.95, txt, va='top', ha='left', fontsize=11)

plt.show()

print('=== 相轨迹簇图 + 吸引域相图 生成完成 ===')
for model in phase_models:
    print(f"{model['name']}: 样本点={model['phase'].shape[0]}, 稳定占比={model['stable_ratio']*100:.1f}%")


In [ ]:
# =========================
# TF12 cargo force analysis
# =========================
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
if 'payload_force_hist' not in globals() or len(payload_force_hist) == 0:
    raise RuntimeError('payload_force_hist is empty. Run the TF12 main cell first.')
force_steps = len(payload_force_hist)
t_force = np.arange(force_steps) * dt
fx_hist = np.array([item['fx_payload'] for item in payload_force_hist], dtype=float)
fy_hist = np.array([item['fy_payload'] for item in payload_force_hist], dtype=float)
mz_hist = np.array([item['mz_payload'] for item in payload_force_hist], dtype=float)
corner_load_hist = np.array([item['corner_normal_loads'] for item in payload_force_hist], dtype=float)
print('=== Cargo force summary ===')
print(payload_force_summary)
print('payload_change_mask =', payload_change_mask)
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle('TF12 货物受力分析', fontsize=14, fontweight='bold')
axes[0].plot(t_force, fx_hist, label='F_x payload', linewidth=1.6)
axes[0].plot(t_force, fy_hist, label='F_y payload', linewidth=1.6)
axes[0].plot(t_force, mz_hist, label='M_z payload', linewidth=1.6)
axes[0].set_ylabel('Force / Moment')
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc='best')
for idx, lab in enumerate(['FL','FR','RL','RR']):
    axes[1].plot(t_force, corner_load_hist[:, idx], label=f'{lab} normal load', linewidth=1.4)
axes[1].set_xlabel('Time [s]')
axes[1].set_ylabel('Normal load [N]')
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc='best', ncol=2)
plt.tight_layout()
plt.show()


In [ ]:
import importlib
import tf12_phase_cluster_raw
importlib.reload(tf12_phase_cluster_raw)

In [ ]:
from tf12_phase_cluster_raw import plot_raw_phase_clusters

raw_phase_summary = plot_raw_phase_clusters(
    main_result=main_result,
    team_state_hist=team_state_hist,
    xt_actual_vehicles=xt_actual_vehicles,
    dt=dt,
    num_vehicles=num_vehicles,
    cfg={
        "traj_only_samples": 900,  # 想更密可以继续加
        "traj_only_len": 140,
        "hist_seed_ratio": 0.45,
    },
    save_path="tf12_raw_phase_clusters.png",
    show=False,
)

raw_phase_summary


In [ ]:
# =========================
# TF12_B1: 对比实验配置（稳定可跑版）
# Baseline: 仅双线性Koopman + AdaptNet（稳定优先）
# Main: 与主文档 METHOD_CFG 严格对齐
# =========================
import copy
import numpy as np

B1_SHOWCASE_PROFILE = {
    # fullpath | stress
    "mode": "fullpath",

    # 主方法严格与当前主文档 METHOD_CFG 对齐
    "main_align_strict": True,

    # baseline 的保底救援（失败时小幅放松保护阈值）
    "enable_baseline_rescue": True,

    # 对比实验是否强制“满程优先”
    "force_fullpath_compare": True,

    # baseline 速度比例（稳定优先建议 < 1.0）
    "baseline_ref_speed_scale": 0.72,

    # baseline 重试时附加降速因子（每次 attempt 叠乘）
    "baseline_retry_slowdown": 0.94,

    # baseline 稳定模式：关闭实时预算跳步，优先可收敛
    "baseline_stability_mode": True,

    # baseline 求解加速开关（仅在你明确要速度时打开）
    "baseline_fast_solver": False,

    # case 重试次数（防止无限拉长）
    "max_attempt_baseline": 3,
    "max_attempt_main": 2,

    # 单 case 超时（秒）
    "per_case_timeout_sec": 900.0,

    # -------- fullpath 档（默认） --------
    "fullpath_eval_snr_db": 28.0,
    "fullpath_uncertainty_mode": "sinusoidal",
    "fullpath_plant_amp": 0.10,
    "fullpath_plant_freq": 1.4,
    "fullpath_completion_tol_s": 1.0,
    "fullpath_log_interval": 100,
    "fullpath_max_extra_steps_baseline": 6200,
    "fullpath_max_extra_steps_main": 4200,

    # -------- stress 档（放大差异） --------
    "stress_eval_snr_db": 14.0,
    "stress_uncertainty_mode": "sinusoidal",
    "stress_plant_amp": 0.22,
    "stress_plant_freq": 1.8,
    "stress_completion_tol_s": 1.0,
    "stress_log_interval": 80,
    "stress_max_extra_steps_baseline": 5600,
    "stress_max_extra_steps_main": 2200,

    # Monte-Carlo 默认种子：快速版
    "mc_use_fast_seed_set": True,
    "mc_seeds_fast": [2026, 2027],
    "mc_seeds_full": [2026, 2027, 2028, 2029, 2030, 2031],
}

# baseline 稳定配置（仅用于对比方法，优先“不断轨”）
B1_BASELINE_STABLE_SOLVER_CFG = {
    "horizon": 20,
    "max_sqp_iters": 2,
    "realtime_mode": True,
    "realtime_budget_ratio": 0.62,
    "realtime_control_budget_sec": 0.014,
    "mpc_decimation_steps": 1,
    "min_solve_vehicles_per_step": 4,
    "mpc_skip_on_budget": False,
    "mpc_min_budget_left_sec": 8.0e-4,
    "fast_max_sqp_iters": 2,
    "fast_time_limit": 6.0e-3,
    "fast_time_limit_min": 1.5e-3,
    "leader_always_solve": True,
    "realtime_disable_online_adapt": False,
    "vehicle_order_leader_first": True,
    "realtime_horizon": 16,
}

# baseline 额外加速配置（仅用于对比方法）
B1_BASELINE_FAST_SOLVER_CFG = {
    "horizon": 16,
    "max_sqp_iters": 1,
    "realtime_mode": True,
    "realtime_budget_ratio": 0.55,
    "mpc_decimation_steps": 6,
    "min_solve_vehicles_per_step": 1,
    "mpc_skip_on_budget": False,
    "mpc_min_budget_left_sec": 2.0e-3,
    "fast_max_sqp_iters": 1,
    "fast_time_limit": 8.0e-4,
    "fast_time_limit_min": 5.0e-4,
    "leader_always_solve": True,
    "realtime_disable_online_adapt": True,
    "vehicle_order_leader_first": True,
    "realtime_horizon": 16,
}

_mode = str(B1_SHOWCASE_PROFILE.get('mode', 'fullpath')).lower().strip()
if _mode not in ('fullpath', 'stress'):
    _mode = 'fullpath'

_common_eval = {
    "eval_snr_db": float(B1_SHOWCASE_PROFILE.get(f"{_mode}_eval_snr_db", 24.0)),
    "uncertainty_mode": str(B1_SHOWCASE_PROFILE.get(f"{_mode}_uncertainty_mode", "sinusoidal")),
    "plant_amp": float(B1_SHOWCASE_PROFILE.get(f"{_mode}_plant_amp", 0.10)),
    "plant_freq": float(B1_SHOWCASE_PROFILE.get(f"{_mode}_plant_freq", 1.4)),
    "enforce_full_path": bool(B1_SHOWCASE_PROFILE.get('force_fullpath_compare', True)),
}

# -------------------------
# Baseline 配置：仅保留 bilinear + AdaptNet
# -------------------------
B1_BASELINE_CFG = dict(METHOD_CFG)
B1_BASELINE_CFG.update({
    "name": "tf12_B1_baseline_bilinear_adaptnet",
    "koopman_structure": "bilinear",
    "use_online_model_adaptation": True,
    "online_adaptation_mode": "bilinear_ridge",

    # 仅保留 bilinear Koopman + AdaptNet
    "use_ppc": False,
    "use_dynamic_ppc": False,
    "use_adaptive_weight": False,
    "use_team_stability_guard": False,
    "use_progress_supervisor": False,
    "use_rigid_coord_correction": False,
    "use_connection_compliance": False,
    "use_legacy_hard_brake": False,

    # 关闭 TF12 通信增强模块
    "use_comm_quality_consensus": False,
    "use_delay_compensation": False,
    "use_comm_constraint_tightening": False,
    "use_comm_degraded_fallback": False,
})
B1_BASELINE_CFG.update(_common_eval)
B1_BASELINE_CFG.update({
    "completion_tol_s": float(B1_SHOWCASE_PROFILE.get(f"{_mode}_completion_tol_s", 1.0)),
    "log_interval": int(B1_SHOWCASE_PROFILE.get(f"{_mode}_log_interval", 100)),
    "max_extra_steps": int(B1_SHOWCASE_PROFILE.get(f"{_mode}_max_extra_steps_baseline", 2600)),
})
if bool(B1_SHOWCASE_PROFILE.get('baseline_stability_mode', True)):
    B1_BASELINE_CFG.update(dict(B1_BASELINE_STABLE_SOLVER_CFG))
if bool(B1_SHOWCASE_PROFILE.get('baseline_fast_solver', False)):
    B1_BASELINE_CFG.update(dict(B1_BASELINE_FAST_SOLVER_CFG))

# -------------------------
# Main 配置：严格与 METHOD_CFG 对齐
# -------------------------
B1_MAIN_CFG = dict(METHOD_CFG)
B1_MAIN_CFG.update({"name": "tf12_B1_main_full"})
B1_MAIN_CFG.update(_common_eval)
B1_MAIN_CFG.update({
    "completion_tol_s": float(B1_SHOWCASE_PROFILE.get(f"{_mode}_completion_tol_s", 1.0)),
    "log_interval": int(B1_SHOWCASE_PROFILE.get(f"{_mode}_log_interval", 100)),
    "max_extra_steps": int(B1_SHOWCASE_PROFILE.get(f"{_mode}_max_extra_steps_main", 700)),
})

# fullpath 档：主方法默认使用中等通信扰动，优先保证完整跑通
if _mode == 'fullpath':
    B1_MAIN_CFG.update({
        "comm_packet_loss_base": 0.01,
        "comm_packet_loss_gain": 0.10,
        "comm_delay_steps_max": 3,
        "comm_delay_bias": 0.30,
        "comm_tighten_max_frac": 0.14,
        "comm_degrade_threshold": 0.42,
    })

# stress 档：只在主方法上加通信扰动，便于展示通信增强收益
if _mode == 'stress':
    B1_MAIN_CFG.update({
        "comm_packet_loss_base": 0.08,
        "comm_packet_loss_gain": 0.55,
        "comm_delay_steps_max": 7,
        "comm_delay_bias": 0.68,
        "comm_tighten_max_frac": 0.42,
        "comm_degrade_threshold": 0.65,
    })

print('B1 profile      :', B1_SHOWCASE_PROFILE)
print('B1 mode         :', _mode)
print('B1 baseline cfg :', B1_BASELINE_CFG['name'])
print('B1 main cfg     :', B1_MAIN_CFG['name'])
print('[B1] main_align_strict =', B1_SHOWCASE_PROFILE.get('main_align_strict', True))
print('[B1] baseline stability mode =', B1_SHOWCASE_PROFILE.get('baseline_stability_mode', True))
print('[B1] baseline fast solver =', B1_SHOWCASE_PROFILE.get('baseline_fast_solver', False))

_b1_path_mode = str(TF12_PATH_CFG.get('active_mode', 'dlc')) if 'TF12_PATH_CFG' in globals() else 'dlc'
print('[B1] current path mode:', _b1_path_mode)
if 'TF12_PATH_LIBRARY' in globals() and _b1_path_mode in TF12_PATH_LIBRARY:
    _meta = TF12_PATH_LIBRARY[_b1_path_mode].get('meta', {})
    print('[B1] path meta:', _meta)


In [ ]:
# =========================
# TF12_B1: 运行两组对比实验（带超时保护）
# =========================
import copy
import importlib
import numpy as np
import time

payload_a1 = importlib.reload(payload_a1)
tf12_runtime = importlib.reload(tf12_runtime)


def _is_baseline_name(name: str) -> bool:
    return 'baseline' in str(name).lower()


def _prepare_case_context(cfg_case, *, attempt=0):
    ctx_case = build_a1_runtime_context()
    cfg_case = dict(cfg_case)

    is_baseline = _is_baseline_name(cfg_case.get('name', ''))

    # 主方法严格对齐当前 METHOD_CFG（仅保留比较相关字段覆写）
    if (not is_baseline) and bool(B1_SHOWCASE_PROFILE.get('main_align_strict', True)):
        keep_keys = {
            'name', 'eval_seed', 'eval_snr_db',
            'uncertainty_mode', 'plant_amp', 'plant_freq',
            'enforce_full_path', 'completion_tol_s', 'max_extra_steps', 'log_interval',
            'horizon', 'max_sqp_iters', 'time_limit',
            'realtime_mode', 'realtime_budget_ratio', 'realtime_control_budget_sec',
            'mpc_decimation_steps', 'min_solve_vehicles_per_step',
            'mpc_skip_on_budget', 'mpc_min_budget_left_sec',
            'fast_max_sqp_iters', 'fast_time_limit', 'fast_time_limit_min',
            'leader_always_solve', 'vehicle_order_leader_first',
            'realtime_disable_online_adapt', 'realtime_horizon',
            # allow B1 main overrides to actually take effect
            'use_ppc', 'use_dynamic_ppc', 'use_adaptive_weight',
            'use_online_model_adaptation', 'online_adaptation_mode',
            'use_connection_compliance', 'use_team_stability_guard',
            'use_progress_supervisor', 'use_rigid_coord_correction',
            'use_comm_quality_consensus', 'use_delay_compensation',
            'use_comm_constraint_tightening', 'use_comm_degraded_fallback',
            'comm_packet_loss_base', 'comm_packet_loss_gain',
            'comm_delay_steps_max', 'comm_delay_bias',
            'comm_tighten_max_frac', 'comm_degrade_threshold',
            'comm_consensus_blend_min', 'comm_consensus_blend_max',
        }
        merged = dict(METHOD_CFG)
        for k in keep_keys:
            if k in cfg_case:
                merged[k] = cfg_case[k]
        cfg_case = merged

    ctx_case['METHOD_CFG'] = cfg_case
    ctx_case['TF12_MODULES_MAIN'] = dict(globals().get('TF12_MODULES_MAIN', globals().get('TF11_MODULES_MAIN', {})))

    # 覆写 runtime seed（用于 Monte-Carlo 公平对比）
    if 'eval_seed' in cfg_case and cfg_case['eval_seed'] is not None:
        rc = dict(ctx_case.get('RUN_CFG', {}))
        rc['seed'] = int(cfg_case['eval_seed'])
        ctx_case['RUN_CFG'] = rc

    # 覆写评估噪声（公平施加到两种方法）
    if 'eval_snr_db' in cfg_case and cfg_case['eval_snr_db'] is not None:
        ctx_case['SNR_DB'] = float(cfg_case['eval_snr_db'])

    baseline_rescue_enabled = bool(B1_SHOWCASE_PROFILE.get('enable_baseline_rescue', True))

    # baseline 可选降速（重试自动附加降速）
    if is_baseline:
        speed_scale = float(B1_SHOWCASE_PROFILE.get('baseline_ref_speed_scale', 1.0))
        retry_slowdown = float(np.clip(B1_SHOWCASE_PROFILE.get('baseline_retry_slowdown', 0.96), 0.85, 1.0))
        speed_scale *= retry_slowdown ** int(max(0, attempt))
        speed_scale = float(np.clip(speed_scale, 0.70, 1.00))
        if speed_scale < 0.999:
            x_ref_local = np.asarray(ctx_case['x_ref_raw'], dtype=float).copy()
            if x_ref_local.shape[0] >= 4:
                x_ref_local[3, :] *= speed_scale  # v_x
            if x_ref_local.shape[0] >= 6:
                x_ref_local[5, :] *= speed_scale  # r_ref ~= v_x * kappa
            ctx_case['x_ref_raw'] = x_ref_local

            # 强制重建参考bundle，避免复用旧速度参考
            ctx_case['A1_COORDINATOR'] = None
            ctx_case['A1_REF_BUNDLE'] = None

    # baseline 运行性补丁：仅调整阈值/限幅，不引入额外功能模块
    if is_baseline and baseline_rescue_enabled:
        mpc_case = dict(ctx_case['MPC_CFG'])
        if attempt <= 0:
            mpc_case.update({
                'emergency_ey_th': 3.20,
                'emergency_epsi_th': 1.55,
                'emergency_brake_ax': -0.10,
                'emergency_vx_cap': 10.0,
                'delta_alpha': 0.80,
                'ax_alpha': 0.72,
                'speed_profile_smooth': 0.90,
            })
            acc_floor = -0.18
            steer_lim_soft = 0.14
            accel_ceil = 1.00
        else:
            mpc_case.update({
                'emergency_ey_th': 3.60,
                'emergency_epsi_th': 1.75,
                'emergency_brake_ax': -0.06,
                'emergency_vx_cap': 11.0,
                'delta_alpha': 0.86,
                'ax_alpha': 0.78,
                'speed_profile_smooth': 0.92,
            })
            acc_floor = -0.12
            steer_lim_soft = 0.15
            accel_ceil = 0.90

        ctx_case['MPC_CFG'] = mpc_case

        umin_case = np.asarray(ctx_case['umin_lin_noadapt'], dtype=float).copy()
        umax_case = np.asarray(ctx_case['umax_lin_noadapt'], dtype=float).copy()

        # 纵向：避免过激制动把车辆“刹停在路上”
        umin_case[1] = max(float(umin_case[1]), float(acc_floor))
        umax_case[1] = min(float(umax_case[1]), float(accel_ceil))

        # 横向：略放宽转角上限，减少中大曲率工况的饱和
        umin_case[0] = min(float(umin_case[0]), -float(steer_lim_soft))
        umax_case[0] = max(float(umax_case[0]), float(steer_lim_soft))

        ctx_case['umin_lin_noadapt'] = umin_case
        ctx_case['umax_lin_noadapt'] = umax_case

    return ctx_case, cfg_case


def _run_tf12_b1_case(case_cfg):
    cfg_base = dict(case_cfg)
    is_baseline = _is_baseline_name(cfg_base.get('name', ''))
    force_fullpath = bool(B1_SHOWCASE_PROFILE.get('force_fullpath_compare', True))
    timeout_sec = float(B1_SHOWCASE_PROFILE.get('per_case_timeout_sec', 420.0))

    if is_baseline:
        max_attempt = int(max(0, B1_SHOWCASE_PROFILE.get('max_attempt_baseline', 2)))
    else:
        max_attempt = int(max(0, B1_SHOWCASE_PROFILE.get('max_attempt_main', 0)))

    start_t = time.perf_counter()
    last_res = None

    for attempt in range(max_attempt + 1):
        elapsed = time.perf_counter() - start_t
        if elapsed > timeout_sec:
            print(f"[B1][timeout] case={cfg_base.get('name')} elapsed={elapsed:.1f}s > {timeout_sec:.1f}s, stop retries")
            break

        cfg_try = dict(cfg_base)

        # baseline 重试时切到更保守解法
        if is_baseline and attempt > 0:
            cfg_try.update({
                'realtime_mode': False,
                'mpc_decimation_steps': 1,
                'min_solve_vehicles_per_step': 4,
                'mpc_skip_on_budget': False,
                'max_sqp_iters': int(max(2, cfg_try.get('max_sqp_iters', 2))),
                'time_limit': float(max(0.04, float(cfg_try.get('time_limit', 0.0) or 0.0))),
                'leader_always_solve': True,
            })

        # main 重试时：保持实时模式但放宽预算，防止单步耗时暴涨
        if (not is_baseline) and attempt > 0:
            cfg_try.update({
                'realtime_mode': True,
                'horizon': int(max(18, cfg_try.get('horizon', 20))),
                'max_sqp_iters': int(max(2, cfg_try.get('max_sqp_iters', 2))),
                'time_limit': float(max(0.035, float(cfg_try.get('time_limit', 0.0) or 0.0))),
                'realtime_budget_ratio': float(max(0.82, cfg_try.get('realtime_budget_ratio', 0.75))),
                'realtime_control_budget_sec': float(max(0.015, cfg_try.get('realtime_control_budget_sec', 0.012) or 0.012)),
                'mpc_decimation_steps': 1,
                'min_solve_vehicles_per_step': 3,
                'mpc_skip_on_budget': True,
                'mpc_min_budget_left_sec': float(min(cfg_try.get('mpc_min_budget_left_sec', 4.0e-4), 2.0e-4)),
                'leader_always_solve': True,
                'realtime_disable_online_adapt': False,
            })

        # 重试时放宽步数与容差，避免长路径提前截断
        if attempt > 0:
            extra_step_add = 900 if is_baseline else 1200
            extra_step_cap_add = 1800 if is_baseline else 2600
            cfg_try['max_extra_steps'] = int(min(
                cfg_try.get('max_extra_steps', 0) + extra_step_add,
                cfg_base.get('max_extra_steps', 0) + extra_step_cap_add
            ))
            cfg_try['completion_tol_s'] = float(max(cfg_try.get('completion_tol_s', 1.0), 1.6))

        ctx_try, cfg_try = _prepare_case_context(cfg_try, attempt=attempt)

        if attempt > 0:
            print(
                f"[B1][retry] case={cfg_try.get('name')} attempt={attempt} "
                f"extra={cfg_try.get('max_extra_steps')}"
            )

        run_t0 = time.perf_counter()
        res = tf12_runtime.run_tf12_main(ctx_try, payload_a1, A1_PAYLOAD_CFG, A1_MAIN_CHANGE_MASK)
        run_elapsed = time.perf_counter() - run_t0
        res['_b1_case_elapsed_sec'] = float(run_elapsed)
        last_res = res

        full_path_ok = bool(res.get('full_path_reached', False))
        solve_skip = int(res.get('mpc_solve_skip_count', 0))
        solve_ok = int(res.get('mpc_solve_success_count', 0))
        print(
            f"[B1][diag] case={cfg_try.get('name')} attempt={attempt} "
            f"full_path={full_path_ok} solve_ok={solve_ok} solve_skip={solve_skip} "
            f"max_lat={res.get('max_lat_global', np.nan):.4f} max_long={res.get('max_long_global', np.nan):.4f}"
        )

        if (not force_fullpath) or full_path_ok:
            return res

        if attempt < max_attempt:
            print(
                f"[B1][warn] not full-path: case={cfg_try.get('name')} "
                f"sim_steps={res.get('sim_steps')} final_s={res.get('final_s_vehicles', [])} -> retry"
            )

    return last_res


print('\n=== [B1] Running baseline (bilinear+AdaptNet only) ===')
b1_result_baseline = _run_tf12_b1_case(B1_BASELINE_CFG)

print('\n=== [B1] Running main (full TF12) ===')
b1_result_main = _run_tf12_b1_case(B1_MAIN_CFG)

compare_results_b1 = {
    'baseline_bilinear_adaptnet': b1_result_baseline,
    'main_tf12_full': b1_result_main,
}

# 默认把主方法结果回填，便于后续单元复用
main_result = b1_result_main
team_state_hist = main_result['team_state_hist']
xt_actual_vehicles = main_result['xt_actual_vehicles']


def _brief_metrics(tag, res):
    print(
        f"[{tag}] rmse_lat_mean={res['rmse_lat_mean']:.4f}, rmse_long_mean={res['rmse_long_mean']:.4f}, "
        f"max_lat={res.get('max_lat_global', np.nan):.4f}, max_long={res.get('max_long_global', np.nan):.4f}, "
        f"full_path={res['full_path_reached']}, sim_steps={res['sim_steps']}, "
        f"eval_snr={res.get('eval_snr_db', np.nan):.1f}, elapsed={res.get('_b1_case_elapsed_sec', np.nan):.1f}s"
    )
    if 'step_time_mean' in res:
        print(
            f"    timing: step_mean={res.get('step_time_mean', np.nan):.4f}s, "
            f"step_max={res.get('step_time_max', np.nan):.4f}s, overrun={res.get('step_overrun_ratio', np.nan):.2%}, "
            f"solve_ok={res.get('mpc_solve_success_count', 0)}, solve_skip={res.get('mpc_solve_skip_count', 0)}"
        )
    if 'comm_summary' in res:
        cs = res['comm_summary']
        print(
            f"    comm: q_mean={cs.get('quality_mean', 1.0):.3f}, "
            f"delay_mean={cs.get('delay_steps_mean', 0.0):.3f}, "
            f"loss_mean={cs.get('loss_ratio_mean', 0.0):.3f}, "
            f"degrade_peak={cs.get('degrade_mix_peak', 0.0):.3f}"
        )


print('\n=== B1 Metrics Summary ===')
_brief_metrics('baseline', b1_result_baseline)
_brief_metrics('main', b1_result_main)

if (not bool(b1_result_baseline.get('full_path_reached', False))) or (not bool(b1_result_main.get('full_path_reached', False))):
    print('\n[B1][NOTE] 当前仍有方法未满程：可先完成 fullpath 可比对照，再切 stress 档做鲁棒性差异展示。')



In [ ]:
# =========================
# TF12_B1: 对比图 + 两张路径跟踪图
# 1) 同图叠加对比（两种方法）
# 2) Baseline 单独路径跟踪图
# 3) Main 单独路径跟踪图
# =========================
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

if 'compare_results_b1' not in globals():
    raise RuntimeError('compare_results_b1 不存在，请先运行 TF12_B1 对比实验单元。')

res_base = compare_results_b1['baseline_bilinear_adaptnet']
res_main = compare_results_b1['main_tf12_full']

labels_v = ['车1 (前左)', '车2 (前右)', '车3 (后左)', '车4 (后右)']
colors_v = ['tab:blue', 'tab:purple', 'tab:green', 'tab:red']

def _to_global_xy(result, v):
    x_hist = np.asarray(result['xt_actual_vehicles'][v], dtype=float)
    n_eval = int(result.get('sim_steps', x_hist.shape[0] - 1)) + 1
    n_eval = min(max(2, n_eval), x_hist.shape[0])
    s_v = np.clip(x_hist[:n_eval, 0], s_ref_path[0], s_ref_path[-1])
    ey_v = x_hist[:n_eval, 1]
    xg, yg = frenet_to_global(s_v, ey_v, s_ref_path, x_path, y_ref_path, psi_ref_path)
    return np.asarray(xg), np.asarray(yg)

def _plot_single_method_path(result, fig_title, line_style='-', marker=None, marker_every=3):
    fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True, sharey=True)
    fig.suptitle(fig_title, fontsize=16, fontweight='bold')
    for v, ax in enumerate(axes.flatten()):
        xg, yg = _to_global_xy(result, v)
        ax.plot(x_path, y_ref_path, color='black', linewidth=2.5, label='货物中心参考')
        ax.plot(xg, yg, color=colors_v[v], linewidth=2.0, linestyle=line_style,
                marker=marker, markevery=marker_every if marker is not None else None,
                alpha=0.95, label=labels_v[v])
        ax.set_title(f'{labels_v[v]} 路径跟踪')
        ax.grid(True, alpha=0.30)
        ax.set_xlabel('x [m]')
        ax.set_ylabel('y [m]')
        ax.legend(fontsize=9, loc='upper right')
    plt.tight_layout(rect=[0, 0.02, 1, 0.96])
    plt.show()

# ---- 图1：同图叠加对比 ----
fig_cmp, axes_cmp = plt.subplots(2, 2, figsize=(16, 10), sharex=True, sharey=True)
fig_cmp.suptitle('TF12_B1 对比：Baseline(双线性Koopman+AdaptNet) vs TF12主方法', fontsize=16, fontweight='bold')

for v, ax in enumerate(axes_cmp.flatten()):
    xg_b, yg_b = _to_global_xy(res_base, v)
    xg_m, yg_m = _to_global_xy(res_main, v)

    ax.plot(x_path, y_ref_path, color='black', linewidth=2.5, label='货物中心参考')
    ax.plot(xg_b, yg_b, color=colors_v[v], linewidth=1.8, linestyle='--', alpha=0.85, label='Baseline')
    ax.plot(xg_m, yg_m, color=colors_v[v], linewidth=2.2, linestyle='-', marker='o', markevery=3, alpha=0.95, label='TF12主方法')

    ax.set_title(f'{labels_v[v]} 路径对比')
    ax.grid(True, alpha=0.30)
    ax.set_xlabel('x [m]')
    ax.set_ylabel('y [m]')
    ax.legend(fontsize=9, loc='upper right')

plt.tight_layout(rect=[0, 0.02, 1, 0.96])
plt.show()

# ---- 图2：Baseline 单独路径图 ----
_plot_single_method_path(
    res_base,
    'TF12_B1 Baseline 路径跟踪图（仅双线性Koopman+AdaptNet）',
    line_style='--',
    marker='s',
    marker_every=4,
)

# ---- 图3：Main 单独路径图 ----
_plot_single_method_path(
    res_main,
    'TF12_B1 主方法路径跟踪图（通信增强 + 稳定一致性增强）',
    line_style='-',
    marker='o',
    marker_every=3,
)


In [ ]:
# =========================
# TF12_B1 误差对比（Baseline vs 主方法）
# =========================
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

if 'compare_results_b1' not in globals():
    raise RuntimeError('compare_results_b1 不存在，请先运行 TF12_B1 对比实验单元。')

res_base = compare_results_b1['baseline_bilinear_adaptnet']
res_main = compare_results_b1['main_tf12_full']

labels_v = ['车1', '车2', '车3', '车4']
colors_v = ['tab:blue', 'tab:purple', 'tab:green', 'tab:red']

def _collect_err(result):
    xh = result['xt_actual_vehicles']
    refh = result['ref_vehicle_histories']
    n_vehicle = min(len(xh), len(refh))
    err_lat = []
    err_long = []
    stats = []
    for v in range(n_vehicle):
        xv = np.asarray(xh[v], dtype=float)
        rv = np.asarray(refh[v], dtype=float)
        n_eval = min(xv.shape[0], rv.shape[0])
        e_s = xv[:n_eval, 0] - rv[:n_eval, 0]
        e_y = xv[:n_eval, 1] - rv[:n_eval, 1]
        valid = np.isfinite(e_s) & np.isfinite(e_y)
        e_s = e_s[valid]
        e_y = e_y[valid]
        err_long.append(e_s)
        err_lat.append(e_y)
        stats.append({
            'vehicle': v + 1,
            'rmse_lat': float(np.sqrt(np.mean(e_y ** 2))) if e_y.size else np.nan,
            'rmse_long': float(np.sqrt(np.mean(e_s ** 2))) if e_s.size else np.nan,
            'max_abs_lat': float(np.max(np.abs(e_y))) if e_y.size else np.nan,
            'max_abs_long': float(np.max(np.abs(e_s))) if e_s.size else np.nan,
        })
    return err_lat, err_long, stats

lat_b, long_b, stats_b = _collect_err(res_base)
lat_m, long_m, stats_m = _collect_err(res_main)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 9), sharex=False)
fig.suptitle('TF12_B1 误差对比：Baseline(双线性Koopman+AdaptNet) vs 主方法', fontsize=15, fontweight='bold')

for v in range(min(len(lat_b), len(lat_m))):
    tb = np.arange(len(lat_b[v])) * dt
    tm = np.arange(len(lat_m[v])) * dt
    ax1.plot(tb, lat_b[v], linestyle='--', color=colors_v[v], linewidth=1.4, alpha=0.85,
             label=f'{labels_v[v]} Baseline')
    ax1.plot(tm, lat_m[v], linestyle='-', color=colors_v[v], linewidth=2.0, alpha=0.95,
             label=f'{labels_v[v]} 主方法')

ax1.axhline(0.0, color='gray', linewidth=0.9, linestyle=':')
ax1.set_title('横向误差 e_y 对比')
ax1.set_ylabel('e_y [m]')
ax1.grid(True, alpha=0.30)
ax1.legend(fontsize=8, ncol=4, loc='upper right')

for v in range(min(len(long_b), len(long_m))):
    tb = np.arange(len(long_b[v])) * dt
    tm = np.arange(len(long_m[v])) * dt
    ax2.plot(tb, long_b[v], linestyle='--', color=colors_v[v], linewidth=1.4, alpha=0.85,
             label=f'{labels_v[v]} Baseline')
    ax2.plot(tm, long_m[v], linestyle='-', color=colors_v[v], linewidth=2.0, alpha=0.95,
             label=f'{labels_v[v]} 主方法')

ax2.axhline(0.0, color='gray', linewidth=0.9, linestyle=':')
ax2.set_title('纵向误差 e_s 对比')
ax2.set_xlabel('时间 [s]')
ax2.set_ylabel('e_s [m]')
ax2.grid(True, alpha=0.30)
ax2.legend(fontsize=8, ncol=4, loc='upper right')

plt.tight_layout(rect=[0, 0.02, 1, 0.96])
plt.show()

print('=== TF12_B1 误差统计对比 ===')
for v in range(min(len(stats_b), len(stats_m))):
    sb = stats_b[v]
    sm = stats_m[v]
    print(
        f"车{v+1}: RMSE_lat {sb['rmse_lat']:.4f} -> {sm['rmse_lat']:.4f}, "
        f"RMSE_long {sb['rmse_long']:.4f} -> {sm['rmse_long']:.4f}, "
        f"Max|lat| {sb['max_abs_lat']:.4f} -> {sm['max_abs_lat']:.4f}, "
        f"Max|long| {sb['max_abs_long']:.4f} -> {sm['max_abs_long']:.4f}"
    )


In [ ]:

# =========================
# TF12_B1 一键双工况对比：DLC + Hairpin（Baseline vs Main）
# 运行前提：先运行到 cell26（确保 B1_BASELINE_CFG / B1_MAIN_CFG / _run_tf12_b1_case 已定义）
# =========================
import numpy as np
import importlib

_required = [
    'B1_BASELINE_CFG', 'B1_MAIN_CFG', '_run_tf12_b1_case',
    'TF12_PATH_LIBRARY', 'TF12_PATH_CFG',
    'standardizer_x_kdnn', 'MPC_CFG', 'RUN_CFG',
    'num_states', 'traj_length', 'vx_nom', 'dt',
    'num_vehicles', 'x0_vehicles',
]
for _name in _required:
    if _name not in globals():
        raise RuntimeError(f'缺少变量: {_name}。请先按顺序运行前置单元。')

payload_a1 = importlib.reload(payload_a1)
tf12_runtime = importlib.reload(tf12_runtime)


B1_DUAL_CFG = globals().get('B1_DUAL_CFG', {
    'modes': ['dlc', 'hairpin'],
    'hairpin_max_extra_baseline': 6200,
    'hairpin_max_extra_main': 7200,
    'hairpin_completion_tol_baseline': 2.0,
    'hairpin_completion_tol_main': 2.0,
})


def _activate_b1_path_mode(mode: str):
    """切换活动路径，并刷新 x_ref_raw / 参考bundle。"""
    global path_length, t_ref, traj_length
    global x_path, y_ref_path, s_ref_path, psi_ref_path, curvature_ref_path
    global x_ref_raw, x_ref_scaled
    global A1_COORDINATOR, A1_REF_BUNDLE
    global x_ref_raw_vehicles, a1_ref_vehicle_histories, x_ref_mpc_vehicles, A1_REFERENCE_VERSION

    mode = str(mode).lower().strip()
    if mode not in TF12_PATH_LIBRARY:
        raise ValueError(f'unknown mode={mode}, available={list(TF12_PATH_LIBRARY.keys())}')

    TF12_PATH_CFG['active_mode'] = mode
    pack = TF12_PATH_LIBRARY[mode]

    path_length = float(pack['path_length'])
    t_ref = pack['t_ref']
    traj_length = int(pack['traj_length'])
    x_path = pack['x_path']
    y_ref_path = pack['y_ref_path']
    s_ref_path = pack['s_ref_path']
    psi_ref_path = pack['psi_ref_path']
    curvature_ref_path = pack['curvature_ref_path']

    # 与 cell8 保持一致：根据曲率构造速度参考
    kappa_abs = np.abs(curvature_ref_path)
    if mode == 'hairpin':
        v_cap = float(np.clip(TF12_PATH_CFG.get('hairpin_ref_speed', 1.35), 1.0, vx_nom))
        curv_gain = float(TF12_PATH_CFG.get('hairpin_speed_profile_gain', 14.0))
        v_min = float(TF12_PATH_CFG.get('hairpin_speed_min', 0.80))
    else:
        v_cap = float(np.clip(TF12_PATH_CFG.get('dlc_speed_cap', vx_nom), 1.0, vx_nom))
        curv_gain = float(MPC_CFG['speed_profile_curv_gain'])
        v_min = float(MPC_CFG['speed_profile_min'])

    vx_ref_profile = v_cap / (1.0 + curv_gain * kappa_abs)
    vx_ref_profile = np.clip(vx_ref_profile, v_min, v_cap)

    smooth_beta = float(np.clip(MPC_CFG['speed_profile_smooth'], 0.0, 0.98))
    for ii in range(1, vx_ref_profile.size):
        vx_ref_profile[ii] = smooth_beta * vx_ref_profile[ii - 1] + (1.0 - smooth_beta) * vx_ref_profile[ii]

    slow_start = float(MPC_CFG['terminal_slowdown_start_s'])
    slow_floor = float(np.clip(MPC_CFG['terminal_slowdown_floor'], 0.4, 1.0))
    if s_ref_path[-1] > slow_start:
        slow_ratio = np.clip((s_ref_path - slow_start) / max(s_ref_path[-1] - slow_start, 1e-6), 0.0, 1.0)
        terminal_scale = 1.0 - (1.0 - slow_floor) * slow_ratio
        vx_ref_profile = np.clip(vx_ref_profile * terminal_scale, max(0.75, 0.85 * v_min), v_cap)

    x_ref_raw = np.zeros((num_states, traj_length))
    x_ref_raw[0, :] = s_ref_path
    x_ref_raw[1, :] = 0.0
    x_ref_raw[2, :] = 0.0
    x_ref_raw[3, :] = vx_ref_profile
    x_ref_raw[4, :] = 0.0
    x_ref_raw[5, :] = vx_ref_profile * curvature_ref_path

    # 纵向对齐：默认把参考 s 起点对齐到当前载荷中心起点，降低 e_s 初始偏置
    s_start_align_cfg = TF12_PATH_CFG.get("s_start_align", None)
    if s_start_align_cfg is None:
        s_start_align = float(x0_payload_center[0])
    else:
        s_start_align = float(s_start_align_cfg)
    if abs(s_start_align) > 1e-12:
        s_ref_path = s_ref_path + s_start_align
        x_ref_raw[0, :] = x_ref_raw[0, :] + s_start_align
        print(f"[TF12][path] applied s_start_align = {s_start_align:.3f} m")

    x_ref_scaled = standardizer_x_kdnn.transform(x_ref_raw.T).T

    # 刷新 A1 参考bundle（避免沿用旧工况参考）
    A1_COORDINATOR, A1_REF_BUNDLE = tf12_runtime.build_a1_reference_bundle(
        payload_module=payload_a1,
        payload_cfg=A1_PAYLOAD_CFG,
        standardizer_x=standardizer_x_kdnn,
        x_ref_raw=x_ref_raw,
        leader_idx=FORMATION_CFG['leader_index'],
        horizon_pad=N_lin_noadapt + 2,
    )
    A1_REFERENCE_VERSION = A1_NOTEBOOK_VERSION
    x_ref_raw_vehicles = A1_REF_BUNDLE['raw_vehicle_refs']
    a1_ref_vehicle_histories = A1_REF_BUNDLE['corner_ref_histories']
    x_ref_mpc_vehicles = A1_REF_BUNDLE['mpc_vehicle_refs']

    return pack


# 双工况自动连跑
B1_DUAL_MODES = [m for m in B1_DUAL_CFG.get('modes', ['dlc', 'hairpin']) if m in TF12_PATH_LIBRARY]
compare_results_b1_dual = {}
summary_rows = []

for mode in B1_DUAL_MODES:
    pack = _activate_b1_path_mode(mode)
    print(f"\n=== [B1-DUAL] mode={mode} | meta={pack.get('meta', {})} ===")

    cfg_base = dict(B1_BASELINE_CFG)
    cfg_main = dict(B1_MAIN_CFG)

    # 回头弯：补充额外步数，但不再用超大值
    if mode == 'hairpin':
        cfg_base['max_extra_steps'] = int(max(cfg_base.get('max_extra_steps', 0), B1_DUAL_CFG.get('hairpin_max_extra_baseline', 2600)))
        cfg_main['max_extra_steps'] = int(max(cfg_main.get('max_extra_steps', 0), B1_DUAL_CFG.get('hairpin_max_extra_main', 2200)))
        cfg_base['completion_tol_s'] = float(max(cfg_base.get('completion_tol_s', 1.8), B1_DUAL_CFG.get('hairpin_completion_tol_baseline', 1.8)))
        cfg_main['completion_tol_s'] = float(max(cfg_main.get('completion_tol_s', 1.8), B1_DUAL_CFG.get('hairpin_completion_tol_main', 1.8)))

        # Hairpin专项：实时稳健模式，避免time-limit连锁与进度停滞
        hairpin_solver_override_main = {
            'realtime_mode': True,
            'horizon': 20,
            'max_sqp_iters': 2,
            'time_limit': 0.035,
            'realtime_budget_ratio': 0.82,
            'realtime_control_budget_sec': 0.015,
            'mpc_decimation_steps': 1,
            'min_solve_vehicles_per_step': 3,
            'mpc_skip_on_budget': True,
            'mpc_min_budget_left_sec': 2.0e-4,
            'fast_max_sqp_iters': 2,
            'fast_time_limit': 6.0e-3,
            'fast_time_limit_min': 1.5e-3,
            'leader_always_solve': True,
            'realtime_disable_online_adapt': False,
            # fullpath对比里取消激进通信退化，避免主方法被“人为卡停”
            'use_comm_constraint_tightening': False,
            'use_comm_degraded_fallback': False,
            'comm_packet_loss_base': 0.00,
            'comm_packet_loss_gain': 0.00,
            'comm_delay_steps_max': 0,
            'comm_delay_bias': 0.00,
            'comm_tighten_max_frac': 0.00,
            'comm_degrade_threshold': 0.05,
        }
        cfg_main.update(hairpin_solver_override_main)

    print('[B1-DUAL] running baseline ...')
    res_b = _run_tf12_b1_case(cfg_base)
    print('[B1-DUAL] running main ...')
    res_m = _run_tf12_b1_case(cfg_main)

    compare_results_b1_dual[mode] = {
        'baseline_bilinear_adaptnet': res_b,
        'main_tf12_full': res_m,
        'path_meta': pack.get('meta', {}),
    }

    for method_name, res in [('baseline', res_b), ('main', res_m)]:
        summary_rows.append({
            'mode': mode,
            'method': method_name,
            'full_path': bool(res.get('full_path_reached', False)),
            'sim_steps': int(res.get('sim_steps', -1)),
            'rmse_lat_mean': float(res.get('rmse_lat_mean', np.nan)),
            'rmse_long_mean': float(res.get('rmse_long_mean', np.nan)),
            'max_lat_global': float(res.get('max_lat_global', np.nan)),
            'max_long_global': float(res.get('max_long_global', np.nan)),
            'elapsed_sec': float(res.get('_b1_case_elapsed_sec', np.nan)),
            'comm_q_mean': float(res.get('comm_summary', {}).get('quality_mean', 1.0)),
        })

# 汇总表
try:
    import pandas as pd
    df_b1_dual = pd.DataFrame(summary_rows)
    df_b1_dual = df_b1_dual.sort_values(['mode', 'method']).reset_index(drop=True)
    print('\n=== TF12_B1 双工况汇总表 ===')
    display(df_b1_dual)
except Exception:
    df_b1_dual = summary_rows
    print('\n=== TF12_B1 双工况汇总表（文本）===')
    for row in df_b1_dual:
        print(row)

# 默认回填到 DLC，兼容后续 cell27/cell28 的 compare_results_b1 画图流程
_default_mode = 'dlc' if 'dlc' in compare_results_b1_dual else list(compare_results_b1_dual.keys())[0]
compare_results_b1 = compare_results_b1_dual[_default_mode]
main_result = compare_results_b1['main_tf12_full']
team_state_hist = main_result['team_state_hist']
xt_actual_vehicles = main_result['xt_actual_vehicles']

print(f"[B1-DUAL] compare_results_b1 已回填为 mode={_default_mode}，可直接复用 cell27/cell28。")





In [ ]:

# =========================
# TF12_B1 Monte-Carlo 汇总（防卡死版）
# 说明：统计成功率 / 均值误差 / 95分位峰值误差
# =========================
import numpy as np
import time

if 'B1_BASELINE_CFG' not in globals() or 'B1_MAIN_CFG' not in globals() or '_run_tf12_b1_case' not in globals():
    raise RuntimeError('请先运行 B1 配置与运行单元（cell25/26）。')

B1_MC_CFG = globals().get('B1_MC_CFG', {
    # 为避免最后一个 cell 长时间卡住，默认仅跑 DLC；需要时可改成 ['dlc','hairpin']
    'modes': ['dlc'],

    # 运行时控制
    'overall_timeout_sec': 1800.0,
    'per_seed_pause_sec': 0.0,

    # MC 阶段默认关闭满程扩展（固定步长，保证可结束）
    'force_nominal_length': True,

    # MC 专用额外步数（仅当 force_nominal_length=False 才生效）
    'dlc_max_extra_baseline': 800,
    'dlc_max_extra_main': 300,
    'hairpin_max_extra_baseline': 1600,
    'hairpin_max_extra_main': 900,
})

if bool(B1_SHOWCASE_PROFILE.get('mc_use_fast_seed_set', True)):
    mc_seeds = list(B1_SHOWCASE_PROFILE.get('mc_seeds_fast', [2026, 2027]))
else:
    mc_seeds = list(B1_SHOWCASE_PROFILE.get('mc_seeds_full', [2026, 2027, 2028, 2029]))

all_modes = B1_MC_CFG.get('modes', ['dlc'])
mc_modes = [m for m in all_modes if m in globals().get('TF12_PATH_LIBRARY', {m: None})]
if len(mc_modes) == 0:
    mc_modes = [str(TF12_PATH_CFG.get('active_mode', 'dlc'))]

rows = []
start_mc = time.perf_counter()
deadline_mc = start_mc + float(B1_MC_CFG.get('overall_timeout_sec', 1800.0))

for mode in mc_modes:
    if '_activate_b1_path_mode' in globals():
        _activate_b1_path_mode(mode)

    for sd in mc_seeds:
        if time.perf_counter() >= deadline_mc:
            print(f"[B1-MC][timeout] overall timeout reached, stop at mode={mode}, seed={sd}")
            break

        cfg_b = dict(B1_BASELINE_CFG)
        cfg_m = dict(B1_MAIN_CFG)
        cfg_b['eval_seed'] = int(sd)
        cfg_m['eval_seed'] = int(sd)

        if bool(B1_MC_CFG.get('force_nominal_length', True)):
            cfg_b['enforce_full_path'] = False
            cfg_m['enforce_full_path'] = False
            cfg_b['max_extra_steps'] = 0
            cfg_m['max_extra_steps'] = 0
        else:
            if mode == 'hairpin':
                cfg_b['max_extra_steps'] = int(B1_MC_CFG.get('hairpin_max_extra_baseline', cfg_b.get('max_extra_steps', 1600)))
                cfg_m['max_extra_steps'] = int(B1_MC_CFG.get('hairpin_max_extra_main', cfg_m.get('max_extra_steps', 900)))
            else:
                cfg_b['max_extra_steps'] = int(B1_MC_CFG.get('dlc_max_extra_baseline', cfg_b.get('max_extra_steps', 800)))
                cfg_m['max_extra_steps'] = int(B1_MC_CFG.get('dlc_max_extra_main', cfg_m.get('max_extra_steps', 300)))

        print(f"[B1-MC] mode={mode}, seed={sd} -> baseline")
        rb = _run_tf12_b1_case(cfg_b)
        print(f"[B1-MC] mode={mode}, seed={sd} -> main")
        rm = _run_tf12_b1_case(cfg_m)

        rows.append({
            'mode': mode,
            'seed': int(sd),
            'method': 'baseline',
            'full_path': bool(rb.get('full_path_reached', False)),
            'rmse_lat_mean': float(rb.get('rmse_lat_mean', np.nan)),
            'rmse_long_mean': float(rb.get('rmse_long_mean', np.nan)),
            'max_lat_global': float(rb.get('max_lat_global', np.nan)),
            'max_long_global': float(rb.get('max_long_global', np.nan)),
            'sim_steps': int(rb.get('sim_steps', -1)),
            'elapsed_sec': float(rb.get('_b1_case_elapsed_sec', np.nan)),
        })
        rows.append({
            'mode': mode,
            'seed': int(sd),
            'method': 'main',
            'full_path': bool(rm.get('full_path_reached', False)),
            'rmse_lat_mean': float(rm.get('rmse_lat_mean', np.nan)),
            'rmse_long_mean': float(rm.get('rmse_long_mean', np.nan)),
            'max_lat_global': float(rm.get('max_lat_global', np.nan)),
            'max_long_global': float(rm.get('max_long_global', np.nan)),
            'sim_steps': int(rm.get('sim_steps', -1)),
            'elapsed_sec': float(rm.get('_b1_case_elapsed_sec', np.nan)),
        })

        pause_t = float(B1_MC_CFG.get('per_seed_pause_sec', 0.0))
        if pause_t > 1e-6:
            time.sleep(pause_t)

    if time.perf_counter() >= deadline_mc:
        break

try:
    import pandas as pd
    df_mc = pd.DataFrame(rows)

    def _q95(x):
        xa = np.asarray(x, dtype=float)
        xa = xa[np.isfinite(xa)]
        if xa.size == 0:
            return np.nan
        return float(np.quantile(xa, 0.95))

    if len(df_mc) == 0:
        print('[B1-MC] 无有效结果（可能被超时提前终止）')
    else:
        summary = (
            df_mc.groupby(['mode', 'method'], as_index=False)
            .agg(
                success_rate=('full_path', 'mean'),
                rmse_lat_mean=('rmse_lat_mean', 'mean'),
                rmse_long_mean=('rmse_long_mean', 'mean'),
                max_lat_p95=('max_lat_global', _q95),
                max_long_p95=('max_long_global', _q95),
                sim_steps_mean=('sim_steps', 'mean'),
                elapsed_mean=('elapsed_sec', 'mean'),
            )
            .sort_values(['mode', 'method'])
            .reset_index(drop=True)
        )

        print('\n=== TF12_B1 Monte-Carlo Summary ===')
        display(summary)

        # 优势量化（main 相对 baseline）
        improve_rows = []
        for mode in sorted(df_mc['mode'].unique()):
            sb = summary[(summary['mode'] == mode) & (summary['method'] == 'baseline')]
            sm = summary[(summary['mode'] == mode) & (summary['method'] == 'main')]
            if len(sb) == 0 or len(sm) == 0:
                continue
            sb = sb.iloc[0]
            sm = sm.iloc[0]

            def _imp(old, new):
                old = float(old)
                new = float(new)
                if not np.isfinite(old) or abs(old) < 1e-12:
                    return np.nan
                return (old - new) / abs(old) * 100.0

            improve_rows.append({
                'mode': mode,
                'success_rate_gain_pp': float((sm['success_rate'] - sb['success_rate']) * 100.0),
                'rmse_lat_improve_pct': _imp(sb['rmse_lat_mean'], sm['rmse_lat_mean']),
                'rmse_long_improve_pct': _imp(sb['rmse_long_mean'], sm['rmse_long_mean']),
                'max_lat_p95_improve_pct': _imp(sb['max_lat_p95'], sm['max_lat_p95']),
                'max_long_p95_improve_pct': _imp(sb['max_long_p95'], sm['max_long_p95']),
                'elapsed_mean_reduce_pct': _imp(sb['elapsed_mean'], sm['elapsed_mean']),
            })

        df_mc_improve = pd.DataFrame(improve_rows)
        print('\n=== 主方法相对 Baseline 的优势（%）===')
        display(df_mc_improve)

except Exception as e:
    print('[B1-MC] pandas unavailable, raw rows:')
    print(rows)
    print('err =', e)



In [ ]:
# =========================
# TF12_B1 增强版误差对比图（单独可运行）
# 横向误差 / 纵向误差分图 + 统计表
# =========================
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

if 'compare_results_b1' not in globals() or not isinstance(compare_results_b1, dict):
    raise RuntimeError('compare_results_b1 不存在，请先运行 B1 对比主单元（cell26）。')
if 'baseline_bilinear_adaptnet' not in compare_results_b1 or 'main_tf12_full' not in compare_results_b1:
    raise RuntimeError('compare_results_b1 缺少 baseline/main 结果。')
if 'dt' not in globals():
    raise RuntimeError('dt 未定义，请先运行前置单元。')

res_base = compare_results_b1['baseline_bilinear_adaptnet']
res_main = compare_results_b1['main_tf12_full']

labels_v = [f'车{i+1}' for i in range(8)]
colors_v = ['tab:blue', 'tab:purple', 'tab:green', 'tab:red', 'tab:orange', 'tab:brown', 'tab:pink', 'tab:gray']

def _collect_err(result):
    xh = result.get('xt_actual_vehicles', [])
    refh = result.get('ref_vehicle_histories', [])
    n_vehicle = min(len(xh), len(refh))
    out = []
    for v in range(n_vehicle):
        xv = np.asarray(xh[v], dtype=float)
        rv = np.asarray(refh[v], dtype=float)
        n_eval = min(xv.shape[0], rv.shape[0])
        if n_eval <= 1:
            out.append(None)
            continue
        e_s = xv[:n_eval, 0] - rv[:n_eval, 0]
        e_y = xv[:n_eval, 1] - rv[:n_eval, 1]
        valid = np.isfinite(e_s) & np.isfinite(e_y)
        e_s = e_s[valid]
        e_y = e_y[valid]
        t = np.arange(e_s.size) * float(dt)
        out.append({
            't': t,
            'e_y': e_y,
            'e_s': e_s,
            'rmse_lat': float(np.sqrt(np.mean(e_y ** 2))) if e_y.size else np.nan,
            'rmse_long': float(np.sqrt(np.mean(e_s ** 2))) if e_s.size else np.nan,
            'max_abs_lat': float(np.max(np.abs(e_y))) if e_y.size else np.nan,
            'max_abs_long': float(np.max(np.abs(e_s))) if e_s.size else np.nan,
        })
    return out

err_b = _collect_err(res_base)
err_m = _collect_err(res_main)
nv = min(len(err_b), len(err_m))
if nv == 0:
    raise RuntimeError('没有可用车辆误差数据。')

# -------- 图1：横向误差 --------
fig1, ax1 = plt.subplots(figsize=(14, 5.8))
for v in range(nv):
    if err_b[v] is None or err_m[v] is None:
        continue
    c = colors_v[v % len(colors_v)]
    lb = labels_v[v] if v < len(labels_v) else f'车{v+1}'
    ax1.plot(err_b[v]['t'], err_b[v]['e_y'], linestyle='--', linewidth=1.5, color=c, alpha=0.85, label=f'{lb} Baseline')
    ax1.plot(err_m[v]['t'], err_m[v]['e_y'], linestyle='-',  linewidth=2.0, color=c, alpha=0.95, label=f'{lb} 主方法')

ax1.axhline(0.0, color='gray', linestyle=':', linewidth=1.0)
ax1.set_title('TF12_B1 横向误差对比（e_y）', fontsize=14, fontweight='bold')
ax1.set_xlabel('时间 [s]')
ax1.set_ylabel('e_y [m]')
ax1.grid(True, alpha=0.30)
ax1.legend(fontsize=8, ncol=4, loc='upper right')
plt.tight_layout()
plt.show()

# -------- 图2：纵向误差 --------
fig2, ax2 = plt.subplots(figsize=(14, 5.8))
for v in range(nv):
    if err_b[v] is None or err_m[v] is None:
        continue
    c = colors_v[v % len(colors_v)]
    lb = labels_v[v] if v < len(labels_v) else f'车{v+1}'
    ax2.plot(err_b[v]['t'], err_b[v]['e_s'], linestyle='--', linewidth=1.5, color=c, alpha=0.85, label=f'{lb} Baseline')
    ax2.plot(err_m[v]['t'], err_m[v]['e_s'], linestyle='-',  linewidth=2.0, color=c, alpha=0.95, label=f'{lb} 主方法')

ax2.axhline(0.0, color='gray', linestyle=':', linewidth=1.0)
ax2.set_title('TF12_B1 纵向误差对比（e_s）', fontsize=14, fontweight='bold')
ax2.set_xlabel('时间 [s]')
ax2.set_ylabel('e_s [m]')
ax2.grid(True, alpha=0.30)
ax2.legend(fontsize=8, ncol=4, loc='upper right')
plt.tight_layout()
plt.show()

# -------- 统计输出 --------
rows = []
for v in range(nv):
    if err_b[v] is None or err_m[v] is None:
        continue
    lb = labels_v[v] if v < len(labels_v) else f'车{v+1}'
    rows.append((
        lb,
        err_b[v]['rmse_lat'], err_m[v]['rmse_lat'],
        err_b[v]['rmse_long'], err_m[v]['rmse_long'],
        err_b[v]['max_abs_lat'], err_m[v]['max_abs_lat'],
        err_b[v]['max_abs_long'], err_m[v]['max_abs_long'],
    ))

print('=== TF12_B1 误差统计（Baseline -> 主方法）===')
for r in rows:
    print(
        f"{r[0]}: RMSE_lat {r[1]:.4f} -> {r[2]:.4f} | "
        f"RMSE_long {r[3]:.4f} -> {r[4]:.4f} | "
        f"Max|lat| {r[5]:.4f} -> {r[6]:.4f} | "
        f"Max|long| {r[7]:.4f} -> {r[8]:.4f}"
    )

In [ ]:
# =========================
# TF12_B1 回头路（Hairpin）位置误差对比图
# 误差定义：e_pos = ||p_actual - p_ref||_2（全局坐标）
# =========================
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

if 'dt' not in globals():
    raise RuntimeError('dt 未定义，请先运行前置单元。')
if 'frenet_to_global' not in globals():
    raise RuntimeError('frenet_to_global 未定义，请先运行路径/坐标转换单元。')
if 'TF12_PATH_LIBRARY' not in globals() or 'hairpin' not in TF12_PATH_LIBRARY:
    raise RuntimeError('未找到 hairpin 路径。请先运行路径库相关单元。')

# 优先使用 cell29 的双工况结果
if 'compare_results_b1_dual' in globals() and isinstance(compare_results_b1_dual, dict) and ('hairpin' in compare_results_b1_dual):
    _hp = compare_results_b1_dual['hairpin']
    if 'baseline_bilinear_adaptnet' not in _hp or 'main_tf12_full' not in _hp:
        raise RuntimeError('compare_results_b1_dual[hairpin] 缺少 baseline/main 结果。')
    res_base = _hp['baseline_bilinear_adaptnet']
    res_main = _hp['main_tf12_full']
else:
    # 退化路径：当前 compare_results_b1 本身就是 hairpin
    if 'compare_results_b1' not in globals() or not isinstance(compare_results_b1, dict):
        raise RuntimeError('未找到 hairpin 结果。请先运行 cell29（一键双工况）或在 hairpin 模式下运行 cell26。')
    if 'baseline_bilinear_adaptnet' not in compare_results_b1 or 'main_tf12_full' not in compare_results_b1:
        raise RuntimeError('compare_results_b1 缺少 baseline/main。')
    mode_now = str(globals().get('TF12_PATH_CFG', {}).get('active_mode', 'dlc')).lower().strip()
    if mode_now != 'hairpin':
        raise RuntimeError('当前不是 hairpin 结果。请先运行 cell29 生成 compare_results_b1_dual[hairpin]。')
    res_base = compare_results_b1['baseline_bilinear_adaptnet']
    res_main = compare_results_b1['main_tf12_full']

pack = TF12_PATH_LIBRARY['hairpin']
s_ref = np.asarray(pack['s_ref_path'], dtype=float)
x_ref = np.asarray(pack['x_path'], dtype=float)
y_ref = np.asarray(pack['y_ref_path'], dtype=float)
psi_ref = np.asarray(pack['psi_ref_path'], dtype=float)

labels_v = ['车1(前左)', '车2(前右)', '车3(后左)', '车4(后右)']
colors_v = ['tab:blue', 'tab:purple', 'tab:green', 'tab:red']


def _collect_pos_err(result):
    xh = result.get('xt_actual_vehicles', [])
    rh = result.get('ref_vehicle_histories', [])
    nv = min(len(xh), len(rh))
    out = []
    for v in range(nv):
        xv = np.asarray(xh[v], dtype=float)
        rv = np.asarray(rh[v], dtype=float)
        n_eval = min(xv.shape[0], rv.shape[0])
        if n_eval <= 1:
            out.append(None)
            continue

        s_act = np.clip(xv[:n_eval, 0], s_ref[0], s_ref[-1])
        ey_act = xv[:n_eval, 1]
        s_r = np.clip(rv[:n_eval, 0], s_ref[0], s_ref[-1])
        ey_r = rv[:n_eval, 1]

        xg_act, yg_act = frenet_to_global(s_act, ey_act, s_ref, x_ref, y_ref, psi_ref)
        xg_ref, yg_ref = frenet_to_global(s_r, ey_r, s_ref, x_ref, y_ref, psi_ref)

        ex = np.asarray(xg_act, dtype=float) - np.asarray(xg_ref, dtype=float)
        ey = np.asarray(yg_act, dtype=float) - np.asarray(yg_ref, dtype=float)
        valid = np.isfinite(ex) & np.isfinite(ey)
        ex = ex[valid]
        ey = ey[valid]
        epos = np.hypot(ex, ey)
        t = np.arange(epos.size) * float(dt)

        out.append({
            't': t,
            'ex': ex,
            'ey': ey,
            'epos': epos,
            'rms_pos': float(np.sqrt(np.mean(epos ** 2))) if epos.size else np.nan,
            'max_pos': float(np.max(np.abs(epos))) if epos.size else np.nan,
            'mean_pos': float(np.mean(epos)) if epos.size else np.nan,
        })
    return out


err_b = _collect_pos_err(res_base)
err_m = _collect_pos_err(res_main)
nv = min(len(err_b), len(err_m), 4)
if nv <= 0:
    raise RuntimeError('hairpin 结果中没有可用车辆位置误差数据。')

# 1) 四车位置误差子图
fig, axes = plt.subplots(2, 2, figsize=(14.5, 9), sharex=True)
fig.suptitle('TF12_B1 回头路位置误差对比（Baseline vs 主方法）', fontsize=15, fontweight='bold')

for v, ax in enumerate(axes.flatten()[:nv]):
    if err_b[v] is None or err_m[v] is None:
        ax.set_title(f'{labels_v[v]} 无有效数据')
        ax.grid(True, alpha=0.3)
        continue

    ax.plot(err_b[v]['t'], err_b[v]['epos'], '--', color=colors_v[v], linewidth=1.6, alpha=0.9, label='Baseline')
    ax.plot(err_m[v]['t'], err_m[v]['epos'], '-', color=colors_v[v], linewidth=2.0, alpha=0.95, label='主方法')
    ax.axhline(0.10, color='gray', linestyle=':', linewidth=1.0, alpha=0.9, label='0.10 m阈值' if v == 0 else None)
    ax.set_title(labels_v[v])
    ax.set_ylabel('位置误差 e_pos [m]')
    ax.grid(True, alpha=0.30)
    ax.legend(fontsize=8, loc='upper right')

for ax in axes[1, :]:
    ax.set_xlabel('时间 [s]')

plt.tight_layout(rect=[0, 0.02, 1, 0.96])
plt.show()

# 2) 车队聚合误差（均值/最大）
Tmax = 0
for v in range(nv):
    if err_b[v] is not None:
        Tmax = max(Tmax, err_b[v]['epos'].size)
    if err_m[v] is not None:
        Tmax = max(Tmax, err_m[v]['epos'].size)

if Tmax > 1:
    def _stack(series_list, key):
        arr = np.full((len(series_list), Tmax), np.nan, dtype=float)
        for i, d in enumerate(series_list):
            if d is None:
                continue
            vec = np.asarray(d[key], dtype=float)
            n = min(Tmax, vec.size)
            arr[i, :n] = vec[:n]
        return arr

    Eb = _stack(err_b[:nv], 'epos')
    Em = _stack(err_m[:nv], 'epos')
    t_common = np.arange(Tmax) * float(dt)

    mean_b = np.nanmean(Eb, axis=0)
    mean_m = np.nanmean(Em, axis=0)
    max_b = np.nanmax(Eb, axis=0)
    max_m = np.nanmax(Em, axis=0)

    plt.figure(figsize=(13.5, 5.2))
    plt.plot(t_common, mean_b, '--', color='tab:gray', linewidth=1.8, label='Baseline 车队均值')
    plt.plot(t_common, mean_m, '-', color='black', linewidth=2.2, label='主方法 车队均值')
    plt.plot(t_common, max_b, '--', color='tab:orange', linewidth=1.6, label='Baseline 车队最大')
    plt.plot(t_common, max_m, '-', color='tab:red', linewidth=2.0, label='主方法 车队最大')
    plt.axhline(0.10, color='gray', linestyle=':', linewidth=1.0, alpha=0.9, label='0.10 m阈值')
    plt.title('TF12_B1 回头路位置误差聚合对比')
    plt.xlabel('时间 [s]')
    plt.ylabel('e_pos [m]')
    plt.grid(True, alpha=0.30)
    plt.legend(loc='upper right', ncol=2)
    plt.tight_layout()
    plt.show()

print('=== Hairpin 位置误差统计（Baseline -> 主方法）===')
for v in range(nv):
    if err_b[v] is None or err_m[v] is None:
        continue
    print(
        f"{labels_v[v]}: "
        f"RMS_pos {err_b[v]['rms_pos']:.4f} -> {err_m[v]['rms_pos']:.4f}, "
        f"Mean_pos {err_b[v]['mean_pos']:.4f} -> {err_m[v]['mean_pos']:.4f}, "
        f"Max_pos {err_b[v]['max_pos']:.4f} -> {err_m[v]['max_pos']:.4f}"
    )

In [ ]:
 # =========================
# TF12 全套对比出图程序（一键）
# 依赖：先运行 cell29 产生 compare_results_b1_dual
# =========================
from tf12_full_compare_figs import generate_tf12_full_comparison_figs

if 'compare_results_b1_dual' not in globals() or not isinstance(compare_results_b1_dual, dict):
    raise RuntimeError('compare_results_b1_dual 不存在，请先运行 cell29（双工况对比）。')
if 'TF12_PATH_LIBRARY' not in globals() or not isinstance(TF12_PATH_LIBRARY, dict):
    raise RuntimeError('TF12_PATH_LIBRARY 不存在，请先运行路径生成单元。')

tf12_full_fig_summary = generate_tf12_full_comparison_figs(
    compare_results_b1_dual=compare_results_b1_dual,
    dt=float(dt),
    tf12_path_library=TF12_PATH_LIBRARY,
    output_dir='results/tf12_full_compare_figs',
    frenet_to_global_fn=frenet_to_global,
    show=False,
)

tf12_full_fig_summary
